# Agentic AI Test Case Generator

### Capstone Project — Google Colab + Groq

**Mandatory single-notebook execution order**

COMMON PROJECT SETUP
↓
FEATURE A — COMPLETE END-TO-END
↓
STOP
↓
FEATURE B — COMPLETE END-TO-END
↓
FINAL COMBINED REPORTING

The existing project flow and agent functions are retained. Acceptance Criteria are externalized only to:
1. `feature_a_acceptance_criteria.json`
2. `feature_b_acceptance_criteria.json`

All implementation, agents, prompts, utilities, execution gates, validation, reporting, logs, summaries, and output-generation logic remain in this one `.ipynb`.

**Execution note:** the supplied capstone does not contain a real application/API/browser SUT. The Test Execution stage therefore runs the existing deterministic QA gate against the generated test-suite artifact and does not fabricate application behavior results.


## 1. Install libraries
Run this cell first.

In [1]:
# Google Colab dependency installation
!pip install -q "groq>=0.31.0" pandas openpyxl json-repair reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.9 MB/s eta 0:00:00


## 2. Imports and Groq connection
In Colab, click the **Secrets (🔑)** icon → add `GROQ_API_KEY` → enable notebook access.

In [2]:
# Colab-safe imports
import os
import json
import re
import time
import pandas as pd

try:
    from google.colab import userdata
except ImportError:
    userdata = None

from groq import Groq


In [3]:
# Secure Groq authentication and stable model configuration
api_key = None
if userdata is not None:
    try:
        api_key = userdata.get("GROQ_API_KEY")
    except Exception:
        api_key = None

api_key = api_key or os.environ.get("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY is required. In Google Colab, open the Secrets (key) panel, "
        "create GROQ_API_KEY, and enable notebook access. Never hardcode the key."
    )

MODEL = "openai/gpt-oss-120b"
client = Groq(api_key=api_key)
available_models = [m.id for m in client.models.list().data]
if MODEL not in available_models:
    raise RuntimeError(
        f"Configured model '{MODEL}' is not available for this API key.\n"
        "Available models include: " + ", ".join(available_models[:30])
    )
print(f"Groq authentication successful. Model: {MODEL}")


Groq authentication successful. Model: openai/gpt-oss-120b


In [4]:
# Optional model availability check
try:
    available_models = [m.id for m in client.models.list().data]
    print("Configured model available:", MODEL in available_models)
    if MODEL not in available_models:
        raise RuntimeError(
            f"Configured model '{MODEL}' is not available for this API key. "
            "Choose an available text-generation model and update MODEL."
        )
except Exception as exc:
    raise RuntimeError(f"Groq model availability check failed: {exc}") from exc


Configured model available: True


## 3. Test the API
If this cell returns an answer, the connection is working.

In [5]:
# Lightweight API smoke test
try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a software testing expert."},
            {"role": "user", "content": "Explain positive testing in one sentence."}
        ],
        temperature=0,
        max_completion_tokens=120
    )
    print(response.choices[0].message.content.strip())
except Exception as exc:
    raise RuntimeError(f"Groq API smoke test failed: {exc}") from exc


Positive testing verifies that a system behaves as expected when given valid, typical inputs and conditions, confirming that it correctly performs its intended functions.


## 4. Reusable JSON call with retry
The agent uses structured JSON responses so the generated test suite can be exported to CSV/Excel/Gherkin.

In [6]:
def _bounded_completion_tokens(prompt, requested, hard_cap=2200):
    """Keep each request safely below an 8K-token request budget."""
    estimated_input_tokens = max(1, len(prompt) // 4)
    safety_budget = 7000
    allowed = max(256, safety_budget - estimated_input_tokens)
    return min(requested, hard_cap, allowed)


def call_json_agent(system_prompt, user_prompt, max_tokens=1800, retries=3):
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            completion_tokens = _bounded_completion_tokens(user_prompt, max_tokens)
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                temperature=0,
                max_completion_tokens=completion_tokens,
            )
            raw = (response.choices[0].message.content or "").strip()
            if not raw:
                raise ValueError("Model returned an empty response.")
            raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.I)
            raw = re.sub(r"\s*```$", "", raw).strip()
            return json.loads(raw)
        except Exception as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(2 * attempt)
    raise RuntimeError(f"JSON agent failed after {retries} attempts: {last_error}") from last_error


def ask_text_agent(system_prompt, user_prompt, max_tokens=1600):
    completion_tokens = _bounded_completion_tokens(user_prompt, max_tokens)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        temperature=0,
        max_completion_tokens=completion_tokens
    )
    return (response.choices[0].message.content or "").strip()


In [7]:
def _estimate_tokens(text):
    """Conservative rough token estimate for rate-limit budgeting."""
    return max(1, len(str(text)) // 4)


# Groq free/on-demand accounts can enforce a rolling tokens-per-minute limit.
# Keep a local rolling budget so Feature A and Feature B do not collide.
REQUEST_TPM_BUDGET = 7000
_recent_token_requests = []


def _wait_for_token_budget(estimated_tokens):
    global _recent_token_requests
    while True:
        now = time.monotonic()
        _recent_token_requests = [
            (ts, tokens) for ts, tokens in _recent_token_requests
            if now - ts < 60
        ]
        used = sum(tokens for _, tokens in _recent_token_requests)

        if used + estimated_tokens <= REQUEST_TPM_BUDGET:
            _recent_token_requests.append((now, estimated_tokens))
            return

        oldest_ts = min(ts for ts, _ in _recent_token_requests)
        sleep_for = max(1.0, 60 - (now - oldest_ts) + 0.5)
        print(f"⏳ Groq TPM pacing: waiting {sleep_for:.1f}s before the next AI call...")
        time.sleep(sleep_for)


def _bounded_completion_tokens(prompt, requested, hard_cap=1400):
    estimated_input_tokens = _estimate_tokens(prompt)
    # Keep input + requested completion comfortably below the 8K limit.
    safety_budget = 6000
    allowed = max(256, safety_budget - estimated_input_tokens)
    return min(requested, hard_cap, allowed)


def ask_ai(prompt, temperature=0, max_tokens=1400, retries=4):
    """Colab-safe Groq call with prompt bounds, TPM pacing, and retry handling."""
    MAX_PROMPT_CHARS = 16000
    if len(prompt) > MAX_PROMPT_CHARS:
        prompt = prompt[:MAX_PROMPT_CHARS] + "\n[Prompt shortened to stay within the API budget.]"

    last_error = None

    for attempt in range(1, retries + 1):
        completion_tokens = _bounded_completion_tokens(
            prompt, max_tokens, hard_cap=1400
        )
        estimated_request_tokens = _estimate_tokens(prompt) + completion_tokens
        _wait_for_token_budget(estimated_request_tokens)

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a senior software QA engineer. "
                            "Create accurate, requirement-traceable software test cases. "
                            "Never invent requirements or functionality."
                        )
                    },
                    {"role": "user", "content": prompt}
                ],
                temperature=temperature,
                max_completion_tokens=completion_tokens
            )

            text = (response.choices[0].message.content or "").strip()
            if not text:
                raise ValueError("AI returned an empty response.")
            return text

        except Exception as exc:
            last_error = exc
            error_text = str(exc).lower()

            # Retry transient Groq rate-limit responses.
            if ("rate_limit" in error_text or "rate limit" in error_text
                    or "429" in error_text or "413" in error_text):
                if attempt < retries:
                    wait_seconds = min(30, 5 * attempt)
                    print(
                        f"⚠️ Groq rate limit encountered. "
                        f"Retrying in {wait_seconds}s (attempt {attempt}/{retries})..."
                    )
                    time.sleep(wait_seconds)
                    continue

            raise

    raise RuntimeError(f"Groq AI call failed after {retries} attempts: {last_error}") from last_error


In [8]:
def safe_text(value, max_chars=7000):
    """
    Safely converts text to string and limits its size
    to prevent oversized API requests.
    """
    if value is None:
        return ""

    text = str(value).strip()

    if len(text) <= max_chars:
        return text

    return text[:max_chars] + "\n...[truncated]"

## Common Agent Definitions

The existing Generator → Critic → Improver → Validator → Structure flow is initialized once and reused by both features.
No feature is invoked in the common setup section.


In [9]:
def generate_test_cases(requirement):

    prompt = f"""
You are a senior QA test case designer.

Read the following requirement carefully:

{requirement}

Generate a comprehensive draft test suite.

Each test case must include:

- Test Case ID
- Acceptance Criteria ID
- Test Scenario
- Preconditions
- Test Data
- Test Steps
- Expected Result
- Category
- Priority
- Risk

Categories:

Positive
Negative
Boundary
Edge

Rules:

1. Every acceptance criterion must be considered.
2. Test cases must be traceable to an AC.
3. Do not invent functionality.
4. Include boundary conditions from the requirement.
5. Include important timing and threshold conditions.
6. Include negative scenarios.
7. Include edge cases.
8. Mark high-risk scenarios appropriately.
"""

    return ask_ai(prompt, max_tokens=1400)

## 7. Critic Agent
The critic checks the draft against every acceptance criterion and identifies gaps.

In [10]:
def critique_test_cases(feature, draft):
    prompt = f"""
You are an expert software QA reviewer.

FEATURE / REQUIREMENT:
{feature}

DRAFT TEST CASES:
{draft}

Review the draft without rewriting the final suite.
For EVERY acceptance criterion, determine coverage and identify missing or weak scenarios.
Identify duplicates, unsupported assumptions, and traceability issues.
Also check Positive, Negative, Boundary, Edge, Priority, Risk, thresholds, timing,
state changes, error messages, session behavior, repeated attempts, and cart recalculation where applicable.

Return a concise QA critique containing:
COVERAGE REPORT
DUPLICATES
UNSUPPORTED ASSUMPTIONS
RISK GAPS
RECOMMENDATIONS

Do not invent new business requirements.
"""
    return ask_ai(prompt, max_tokens=1200)


## 8. Improver Agent
This is the second pass. It uses critic feedback to fill coverage gaps.

In [11]:
def improve_test_cases(requirement, draft, critique):
    prompt = f"""
You are a senior QA architect.

Requirement:
{safe_text(requirement, 7000)}

Draft Test Suite:
{safe_text(draft, 6500)}

Critic Review:
{safe_text(critique, 4500)}

Create the complete improved final test suite.
Rules:
1. Preserve valid existing test cases.
2. Add missing scenarios identified by the critic.
3. Remove duplicate cases and unsupported assumptions.
4. Maintain AC traceability.
5. Include Positive, Negative, Boundary and Edge cases when supported.
6. Maintain priority and risk.
7. Do not invent functionality.
8. Every acceptance criterion must have at least one traceable test case.
9. Keep each test case concise.
10. Return only the complete test suite content.
"""
    return ask_ai(prompt, max_tokens=1400)


## 9. Validator Agent
The validator gives an explicit PASS/FAIL coverage result.

In [12]:
def validate_test_suite(requirement, final_suite):
    prompt = f"""
You are the final QA test lead.

Requirement:
{requirement}

Final Test Suite:
{final_suite}

Validate the suite concisely. For every acceptance criterion, state whether it is covered
and identify any missing scenario. Also check Positive, Negative, Boundary, Edge,
traceability, risk, duplicates, and unsupported assumptions.

Return:
FINAL COVERAGE STATUS
AC1: PASS/FAIL
AC2: PASS/FAIL
...
FINAL STATUS: PASS/FAIL
If FAIL, list exact missing scenarios.
"""
    return ask_ai(prompt, max_tokens=900)


## Common Structuring, Parsing, Validation and Reporting Tools

These tools are defined before either feature starts so the same existing implementation is reused consistently.


In [13]:
TEST_COLUMNS = [
    "Test Case ID",
    "Feature",
    "Acceptance Criteria",
    "Test Scenario",
    "Preconditions",
    "Test Data",
    "Test Steps",
    "Expected Result",
    "Category",
    "Priority",
    "Risk"
]

print(TEST_COLUMNS)

['Test Case ID', 'Feature', 'Acceptance Criteria', 'Test Scenario', 'Preconditions', 'Test Data', 'Test Steps', 'Expected Result', 'Category', 'Priority', 'Risk']


In [14]:
def structure_test_cases(requirement, final_suite, feature_name):
    prompt = f"""
Convert the final QA test suite below into ONLY a JSON array.

Feature:
{feature_name}

Requirement:
{requirement}

Final Test Suite:
{final_suite}

Each object MUST contain exactly these fields:
Test Case ID, Feature, Acceptance Criteria, Test Scenario, Preconditions,
Test Data, Test Steps, Expected Result, Category, Priority, Risk.

Allowed Category: Positive, Negative, Boundary, Edge
Allowed Priority: P0, P1, P2, P3
Allowed Risk: High, Medium, Low

The Acceptance Criteria value must be an AC identifier present in the requirement.
Do not invent functionality.

IMPORTANT:
- Return the COMPLETE test suite.
- Do not stop early.
- Do not use markdown.
- Do not add explanations.
- Return ONLY valid JSON.
- The response MUST start with [ and end with ].
"""

    if len(prompt) > 20000:
        prompt = prompt[:20000]

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a QA test data formatting specialist. "
                    "Return only complete, valid JSON."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_completion_tokens=3500
    )

    text = (response.choices[0].message.content or "").strip()

    if not text:
        raise ValueError(
            f"{feature_name}: formatter returned an empty response."
        )

    # Detect incomplete JSON immediately
    if not text.endswith("]"):
        raise ValueError(
            f"{feature_name}: formatter returned incomplete JSON. "
            f"Response length={len(text)} characters."
        )

    return text

In [15]:
from json_repair import repair_json

def extract_json_array(text):
    if not text or not str(text).strip():
        raise ValueError("Model returned an empty response.")

    text = str(text).strip()
    text = re.sub(r"```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text).strip()

    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end < start:
        raise ValueError("No complete JSON array found in model response.")

    json_text = text[start:end + 1]

    try:
        repaired = repair_json(json_text, return_objects=False)
        data = json.loads(repaired)
    except Exception as exc:
        raise ValueError(
            f"Could not parse/repair model JSON: {exc}\nResponse excerpt:\n{json_text[:3000]}"
        ) from exc

    if not isinstance(data, list):
        raise ValueError(f"Expected a JSON array, received {type(data).__name__}.")
    return data


In [16]:
def normalize_category(value):
    value = str(value).strip().lower()

    mapping = {
        "positive": "Positive",
        "positive test": "Positive",
        "negative": "Negative",
        "negative test": "Negative",
        "boundary": "Boundary",
        "boundary test": "Boundary",
        "edge": "Edge",
        "edge case": "Edge"
    }

    return mapping.get(value, str(value).strip())


def normalize_priority(value):
    value = str(value).strip().upper()

    if value in ["P0", "P1", "P2", "P3"]:
        return value

    return value


def normalize_risk(value):
    value = str(value).strip().lower()

    mapping = {
        "high": "High",
        "medium": "Medium",
        "low": "Low"
    }

    return mapping.get(value, str(value).strip())

In [17]:
def check_required_columns(df):
    required_columns = [
        "Test Case ID",
        "Feature",
        "Acceptance Criteria",
        "Category",
        "Test Scenario",
        "Preconditions",
        "Test Steps",
        "Test Data",
        "Expected Result"
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        return {
            "status": "FAIL",
            "missing_columns": missing_columns,
            "available_columns": list(df.columns)
        }

    return {
        "status": "PASS",
        "missing_columns": [],
        "available_columns": list(df.columns)
    }

In [18]:
def check_duplicate_ids(df):

    duplicates = df[
        df["Test Case ID"].duplicated(keep=False)
    ]["Test Case ID"].tolist()

    return {
        "PASS": len(duplicates) == 0,
        "Duplicate IDs": sorted(set(duplicates))
    }

In [19]:
VALID_CATEGORIES = {
    "Positive",
    "Negative",
    "Boundary",
    "Edge"
}

def check_categories(df):

    # Get category column
    if "Category" not in df.columns:
        return {
            "PASS": True,
            "Invalid Categories": []
        }

    # Remove NaN and empty values before checking
    categories = (
        df["Category"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    # Remove blank values
    categories = categories[
        categories != ""
    ]

    # Remove string versions of NaN
    categories = categories[
        ~categories.str.lower().isin(["nan", "none", "null"])
    ]

    # Find invalid categories
    invalid_categories = sorted(
        set(categories) - VALID_CATEGORIES
    )

    return {
        "PASS": len(invalid_categories) == 0,
        "Invalid Categories": invalid_categories
    }

In [20]:
VALID_PRIORITIES = {
    "P0",
    "P1",
    "P2",
    "P3"
}

def check_priorities(df):

    if "Priority" not in df.columns:
        return {
            "PASS": True,
            "Invalid Priorities": []
        }

    priorities = (
        df["Priority"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    # Ignore empty values
    priorities = priorities[
        priorities != ""
    ]

    # Ignore string representations of missing values
    priorities = priorities[
        ~priorities.str.lower().isin(
            ["nan", "none", "null"]
        )
    ]

    invalid = sorted(
        set(priorities) - VALID_PRIORITIES
    )

    return {
        "PASS": len(invalid) == 0,
        "Invalid Priorities": invalid
    }

In [21]:
VALID_RISKS = {
    "High",
    "Medium",
    "Low"
}

def check_risks(df):

    if "Risk" not in df.columns:
        return {
            "PASS": True,
            "Invalid Risks": []
        }

    risks = (
        df["Risk"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    # Ignore empty values
    risks = risks[risks != ""]

    # Ignore missing-value strings
    risks = risks[
        ~risks.str.lower().isin(
            ["nan", "none", "null"]
        )
    ]

    invalid = sorted(
        set(risks) - VALID_RISKS
    )

    return {
        "PASS": len(invalid) == 0,
        "Invalid Risks": invalid
    }

In [22]:
def check_blank_values(df):

    columns_to_check = [
        "Test Case ID",
        "Acceptance Criteria",
        "Test Scenario",
        "Expected Result",
        "Category",
        "Priority",
        "Risk"
    ]

    problems = {}

    for col in columns_to_check:
        blank_rows = df[
            df[col].isna() |
            (df[col].astype(str).str.strip() == "")
        ].index.tolist()

        if blank_rows:
            problems[col] = blank_rows

    return {
        "PASS": len(problems) == 0,
        "Blank Fields": problems
    }

In [23]:
import re

def check_ac_format(df):

    invalid = []

    for ac in df["Acceptance Criteria"].dropna().astype(str):

        ac = ac.strip()

        # Ignore empty / missing values
        if ac == "" or ac.lower() in ["nan", "none", "null"]:
            continue

        # Validate only non-empty values
        if not re.fullmatch(r"AC\d+", ac):
            invalid.append(ac)

    return {
        "PASS": len(invalid) == 0,
        "Invalid AC References": sorted(set(invalid))
    }

In [24]:
def coverage_check(df, expected_acs):

    actual_acs = set(
        df["Acceptance Criteria"]
        .astype(str)
        .str.strip()
    )

    expected = set(expected_acs)

    covered = sorted(expected & actual_acs)
    missing = sorted(expected - actual_acs)
    unexpected = sorted(actual_acs - expected)

    return {
        "Total ACs": len(expected),
        "Covered ACs": len(covered),
        "Missing ACs": missing,
        "Unexpected ACs": unexpected,
        "Coverage %": round(
            len(covered) / len(expected) * 100,
            2
        ),
        "PASS": len(missing) == 0
    }

In [25]:
def category_coverage(df):

    required = {
        "Positive",
        "Negative",
        "Boundary",
        "Edge"
    }

    actual = set(df["Category"])

    missing = sorted(required - actual)

    return {
        "PASS": len(missing) == 0,
        "Missing Categories": missing
    }

In [26]:
def check_test_case_count(df, minimum=10):

    count = len(df)

    return {
        "Count": count,
        "Minimum Expected": minimum,
        "PASS": count >= minimum
    }

In [27]:
def run_deterministic_checks(df, expected_acs):

    results = {}

    results["Required Columns"] = check_required_columns(df)
    results["Duplicate IDs"] = check_duplicate_ids(df)
    results["Categories"] = check_categories(df)
    results["Priorities"] = check_priorities(df)
    results["Risks"] = check_risks(df)
    results["Blank Values"] = check_blank_values(df)
    results["AC Format"] = check_ac_format(df)
    results["AC Coverage"] = coverage_check(df, expected_acs)
    results["Category Coverage"] = category_coverage(df)
    results["Test Count"] = check_test_case_count(df)

    return results

In [28]:
def checks_to_dataframe(checks):

    rows = []

    for check_name, result in checks.items():

        rows.append({
            "Check": check_name,
            "Status": "PASS" if result.get("PASS") else "FAIL",
            "Details": str(result)
        })

    return pd.DataFrame(rows)

In [29]:
def create_coverage_report(
    df,
    expected_acs,
    feature_name
):

    rows = []

    for ac in expected_acs:

        matching = df[
            df["Acceptance Criteria"] == ac
        ]

        count = len(matching)

        if count == 0:
            status = "MISSING"
        else:
            status = "COVERED"

        rows.append({
            "Feature": feature_name,
            "Acceptance Criteria": ac,
            "Coverage Status": status,
            "Test Case Count": count,
            "Test Case IDs": ", ".join(
                matching["Test Case ID"].astype(str)
            )
        })

    return pd.DataFrame(rows)

In [30]:
def dataframe_to_gherkin(df, feature_name):

    lines = []

    lines.append(f"Feature: {feature_name}")
    lines.append("")

    for _, row in df.iterrows():

        lines.append(
            f"  Scenario: {row['Test Scenario']}"
        )

        preconditions = str(
            row["Preconditions"]
        ).strip()

        test_data = str(
            row["Test Data"]
        ).strip()

        steps = str(
            row["Test Steps"]
        ).strip()

        expected = str(
            row["Expected Result"]
        ).strip()

        if preconditions:
            lines.append(
                f"    Given {preconditions}"
            )

        if test_data:
            lines.append(
                f"    And the test data is {test_data}"
            )

        for step in steps.split("\n"):

            step = step.strip()

            if step:
                lines.append(
                    f"    When {step}"
                )

        lines.append(
            f"    Then {expected}"
        )

        lines.append("")

    return "\n".join(lines)

In [31]:
def deterministic_qa_check(df, feature_name):
    required_columns = TEST_COLUMNS
    results = []

    missing_columns = [c for c in required_columns if c not in df.columns]
    results.append({
        "Feature": feature_name,
        "Check": "Required Columns",
        "Status": "PASS" if not missing_columns else "FAIL",
        "Details": "All required columns present" if not missing_columns else str(missing_columns)
    })
    if missing_columns:
        return pd.DataFrame(results)

    def nonblank(series):
        return ~series.isna() & ~series.astype(str).str.strip().str.lower().isin(["", "nan", "none", "null"])

    blank_counts = {
        col: int((~nonblank(df[col])).sum())
        for col in required_columns
    }
    mandatory_blank_total = sum(blank_counts.values())
    results.append({
        "Feature": feature_name,
        "Check": "Empty Required Fields",
        "Status": "PASS" if mandatory_blank_total == 0 else "FAIL",
        "Details": "No empty required fields" if mandatory_blank_total == 0 else str(blank_counts)
    })

    duplicates = int(df["Test Case ID"].duplicated().sum())
    results.append({
        "Feature": feature_name,
        "Check": "Duplicate Test Case IDs",
        "Status": "PASS" if duplicates == 0 else "FAIL",
        "Details": f"{duplicates} duplicate rows"
    })

    valid_categories = {"Positive", "Negative", "Boundary", "Edge"}
    actual_categories = set(df["Category"].dropna().astype(str).str.strip())
    invalid_categories = sorted(actual_categories - valid_categories)
    results.append({
        "Feature": feature_name,
        "Check": "Valid Test Categories",
        "Status": "PASS" if not invalid_categories else "FAIL",
        "Details": "All categories valid" if not invalid_categories else str(invalid_categories)
    })

    valid_priorities = {"P0", "P1", "P2", "P3"}
    actual_priorities = set(df["Priority"].dropna().astype(str).str.strip().str.upper())
    actual_priorities -= {"", "NAN", "NONE", "NULL"}
    invalid_priorities = sorted(actual_priorities - valid_priorities)
    results.append({
        "Feature": feature_name,
        "Check": "Valid Priorities",
        "Status": "PASS" if not invalid_priorities else "FAIL",
        "Details": "All priorities valid" if not invalid_priorities else str(invalid_priorities)
    })

    valid_risks = {"High", "Medium", "Low"}
    actual_risks = set(df["Risk"].dropna().astype(str).str.strip())
    actual_risks -= {"", "nan", "none", "null"}
    invalid_risks = sorted(actual_risks - valid_risks)
    results.append({
        "Feature": feature_name,
        "Check": "Valid Risks",
        "Status": "PASS" if not invalid_risks else "FAIL",
        "Details": "All risks valid" if not invalid_risks else str(invalid_risks)
    })

    return pd.DataFrame(results)


In [32]:
# Minimal addition required to externalize Acceptance Criteria without changing the agent flow.
def load_acceptance_criteria(filename, feature_name):
    if not os.path.exists(filename):
        raise FileNotFoundError(
            f"{filename} not found. Upload the editable Acceptance Criteria JSON file."
        )
    with open(filename, encoding="utf-8") as f:
        payload = json.load(f)

    ac_list = payload.get("acceptance_criteria")
    if not isinstance(ac_list, list) or not ac_list:
        raise ValueError(f"{filename} must contain a non-empty 'acceptance_criteria' list.")

    for item in ac_list:
        if "id" not in item:
            raise ValueError(f"{filename} contains an Acceptance Criterion without an id.")
        if "description" not in item and "text" not in item:
            raise ValueError(f"{filename} must provide 'description' or 'text' for every AC.")

    ids = [str(item["id"]).strip() for item in ac_list]
    if len(ids) != len(set(ids)):
        raise ValueError(f"{filename} contains duplicate Acceptance Criteria IDs.")

    print(f"Loaded {len(ac_list)} Acceptance Criteria for {feature_name}.")
    return ac_list

def format_acceptance_criteria(ac_list):
    return "\n".join(
        f"{item['id']}: {item.get('description', item.get('text', ''))}"
        for item in ac_list
    )

def build_requirement_context(requirement, ac_list):
    return (
        requirement.strip()
        + "\n\nAcceptance Criteria (loaded from external editable JSON):\n"
        + format_acceptance_criteria(ac_list)
    )


# FEATURE A — User Login

**Mandatory execution order:** Feature A is executed completely through its final summary before the next feature begins.


## Feature Requirement

In [33]:
feature_a_requirement = 'feature_a = """\nFeature: User Login\n\nUser Story:\nAs a registered user, I want to log in with my email and password\nso that I can access my account.\n\nDescription:\nThe login screen has an Email field, a Password field, a Log In button,\nand a Forgot password? link. On successful login the user is taken to\ntheir dashboard. On failure an error message is shown.'
print(feature_a_requirement)


feature_a = """
Feature: User Login

User Story:
As a registered user, I want to log in with my email and password
so that I can access my account.

Description:
The login screen has an Email field, a Password field, a Log In button,
and a Forgot password? link. On successful login the user is taken to
their dashboard. On failure an error message is shown.


## Acceptance Criteria — Loaded from External Editable File

In [34]:
# ============================================================
# UPLOAD EXTERNAL ACCEPTANCE CRITERIA FILES
# ============================================================

import os
from google.colab import files

REQUIRED_AC_FILES = [
    "feature_a_acceptance_criteria.json",
    "feature_b_acceptance_criteria.json"
]

missing_files = [
    f for f in REQUIRED_AC_FILES
    if not os.path.exists(f)
]

if missing_files:
    print("Please upload the following Acceptance Criteria files:")
    for f in missing_files:
        print(" -", f)

    uploaded = files.upload()

    still_missing = [
        f for f in REQUIRED_AC_FILES
        if not os.path.exists(f)
    ]

    if still_missing:
        raise FileNotFoundError(
            f"Missing Acceptance Criteria files: {still_missing}"
        )

print("✅ Acceptance Criteria files are available.")
print("Feature A:", os.path.exists("feature_a_acceptance_criteria.json"))
print("Feature B:", os.path.exists("feature_b_acceptance_criteria.json"))

Please upload the following Acceptance Criteria files:
 - feature_a_acceptance_criteria.json
 - feature_b_acceptance_criteria.json


Saving feature_a_acceptance_criteria.json to feature_a_acceptance_criteria.json
Saving feature_b_acceptance_criteria.json to feature_b_acceptance_criteria.json
✅ Acceptance Criteria files are available.
Feature A: True
Feature B: True


## Feature Data / Input Preparation

In [35]:
feature_a_ac = load_acceptance_criteria(
    "feature_a_acceptance_criteria.json",
    "User Login"
)

EXPECTED_AC_A = [item["id"] for item in feature_a_ac]

print("Expected AC IDs:", EXPECTED_AC_A)
display(pd.DataFrame(feature_a_ac))

Loaded 9 Acceptance Criteria for User Login.
Expected AC IDs: ['AC1', 'AC2', 'AC3', 'AC4', 'AC5', 'AC6', 'AC7', 'AC8', 'AC9']


,id,title,text
0,AC1,Valid login.,"Given a registered, active user, when they ent..."
1,AC2,Invalid password.,"Given a registered user, when they enter a cor..."
2,AC3,Unregistered email.,When an email that is not registered is entere...
3,AC4,Empty fields.,When either field is left blank and Log In is ...
4,AC5,Email format.,When the email field contains a value that is ...
5,AC6,Account lockout.,After 5 consecutive failed attempts within 15 ...
6,AC7,Case sensitivity.,The email is case-insensitive (User@x.com == u...
7,AC8,Session.,On successful login a session is established; ...
8,AC9,Inactive account.,"A user whose account is deactivated sees ""This..."


In [36]:
feature_a_requirement_context = build_requirement_context(feature_a_requirement, feature_a_ac)
print(feature_a_requirement_context)


feature_a = """
Feature: User Login

User Story:
As a registered user, I want to log in with my email and password
so that I can access my account.

Description:
The login screen has an Email field, a Password field, a Log In button,
and a Forgot password? link. On successful login the user is taken to
their dashboard. On failure an error message is shown.

Acceptance Criteria (loaded from external editable JSON):
AC1: Given a registered, active user, when they enter their correct email and password and click Log In, then they are redirected to the dashboard.
AC2: Given a registered user, when they enter a correct email but wrong password, then an error "Invalid email or password" is shown and they remain on the login page.
AC3: When an email that is not registered is entered with any password, then the same generic "Invalid email or password" error is shown (no indication of whether the email exists).
AC4: When either field is left blank and Log In is clicked, then inline validation p

## Feature Agent Prompt / Invocation / Scenario Generation

In [37]:
draft_a = generate_test_cases(feature_a_requirement_context)
print(draft_a)


**Draft Test Suite – User Login Feature**  
*(All test cases are traceable to the Acceptance Criteria (AC) listed in the requirement. No functionality outside the specification is assumed.)*  

| # | Test Case ID | Acceptance Criteria ID | Test Scenario | Preconditions | Test Data | Test Steps | Expected Result | Category | Priority | Risk |
|---|--------------|------------------------|---------------|---------------|-----------|------------|-----------------|----------|----------|------|
| 1 | TC‑001 | AC1 | Successful login with valid credentials | User **active** and **registered**; not logged in | Email: `user@example.com`  <br>Password: `CorrectPass123!` | 1. Navigate to Login page  <br>2. Enter email  <br>3. Enter password  <br>4. Click **Log In** | User is redirected to the Dashboard; a session cookie is created | Positive | High | Medium |
| 2 | TC‑002 | AC2 | Login fails with correct email but wrong password | User **active** and **registered**; not locked | Email: `user@examp

## Feature Test Case Generation

In [38]:
print("Generated draft test suite for User Login.")
print(draft_a)


Generated draft test suite for User Login.
**Draft Test Suite – User Login Feature**  
*(All test cases are traceable to the Acceptance Criteria (AC) listed in the requirement. No functionality outside the specification is assumed.)*  

| # | Test Case ID | Acceptance Criteria ID | Test Scenario | Preconditions | Test Data | Test Steps | Expected Result | Category | Priority | Risk |
|---|--------------|------------------------|---------------|---------------|-----------|------------|-----------------|----------|----------|------|
| 1 | TC‑001 | AC1 | Successful login with valid credentials | User **active** and **registered**; not logged in | Email: `user@example.com`  <br>Password: `CorrectPass123!` | 1. Navigate to Login page  <br>2. Enter email  <br>3. Enter password  <br>4. Click **Log In** | User is redirected to the Dashboard; a session cookie is created | Positive | High | Medium |
| 2 | TC‑002 | AC2 | Login fails with correct email but wrong password | User **active** and **re

## Feature Critic Agent

In [39]:
critique_a = critique_test_cases(feature_a_requirement_context, draft_a)
print(critique_a)


**QA CRITIQUE – User‑Login Feature (as written)**  

---

## 1. COVERAGE REPORT (per Acceptance Criterion)

| AC | Covered by Draft TC(s) | Gaps / Weaknesses |
|----|------------------------|-------------------|
| **AC1** – successful login | TC‑001 | – No check that the **email is case‑insensitive** (see AC7). <br>– No verification of **session cookie** creation or that the session survives a page‑refresh (AC8). |
| **AC2** – wrong password | TC‑002 | – Does not prove **password case‑sensitivity** (AC7). |
| **AC3** – unknown email | TC‑003 | ✔︎ Fully covered. |
| **AC4** – blank‑field validation | TC‑004 (both blank) <br>TC‑005 (email blank) | – **Password‑blank only** scenario missing. <br>– No explicit step to verify *“no network request is sent”* (e.g., mock or dev‑tools capture). |
| **AC5** – email format | TC‑006 (invalid) <br>TC‑007 (max‑length valid) | – Edge‑case formats (e.g., sub‑domain, plus‑address) are not required but could be useful. <br>– Assumes a UI‑enforced 254‑ch

## Feature Improver Agent

In [40]:
final_a = improve_test_cases(feature_a_requirement_context, draft_a, critique_a)
print(final_a)


⏳ Groq TPM pacing: waiting 46.4s before the next AI call...
**Complete Test Suite – User Login Feature**  
*(All test cases are traceable to the Acceptance Criteria (AC). No functionality outside the specification is assumed.)*  

| # | Test Case ID | Acceptance Criteria ID | Test Scenario | Preconditions | Test Data | Test Steps | Expected Result | Category | Priority | Risk |
|---|--------------|------------------------|---------------|---------------|-----------|------------|-----------------|----------|----------|------|
| 1 | TC‑001 | AC1 | Successful login with valid credentials | Active, registered user; not logged in | Email: `user@example.com`  <br>Password: `CorrectPass123!` | 1. Open Login page  <br>2. Enter email  <br>3. Enter password  <br>4. Click **Log In** | User is redirected to Dashboard; a session cookie is created | Positive | High | Medium |
| 2 | TC‑002 | AC2 | Login fails with correct email but wrong password | Active, registered user; not locked | Email: `user@e

## Feature Validator Agent

In [41]:
validation_a = validate_test_suite(feature_a_requirement_context, final_a)
print(validation_a)


⏳ Groq TPM pacing: waiting 1.8s before the next AI call...
**FINAL COVERAGE STATUS**

| Acceptance Criteria ID | Covered by Test Case(s) | Coverage Verdict |
|------------------------|--------------------------|------------------|
| **AC1** | TC‑001 | **PASS** |
| **AC2** | TC‑002 | **PASS** |
| **AC3** | – | **FAIL** |
| **AC4** | – | **FAIL** |
| **AC5** | – | **FAIL** |
| **AC6** | – | **FAIL** |
| **AC7** | – | **FAIL** |
| **AC8** | – | **FAIL** |
| **AC9** | – | **FAIL** |

**FINAL STATUS:** **FAIL**

---

### Missing / Incomplete Scenarios (per failed AC)

| AC | Missing Test Scenario(s) | Suggested Test Case ID(s) |
|----|--------------------------|---------------------------|
| **AC3** | Verify that entering an **unregistered email** (any password) shows the generic “Invalid email or password” error and stays on the login page. | TC‑003 |
| **AC4** | a) Both fields blank → inline required‑field prompts, no network request.<br>b) Email blank, password filled → email required pr

## Feature Final Test Case Structuring

In [42]:
structured_a_text = structure_test_cases(feature_a_requirement_context, final_a, "User Login")
print(structured_a_text[:3000])


[
  {
    "Test Case ID": "TC-001",
    "Feature": "User Login",
    "Acceptance Criteria": "AC1",
    "Test Scenario": "Successful login with valid credentials",
    "Preconditions": "Active, registered user; not logged in",
    "Test Data": "Email: user@example.com\nPassword: CorrectPass123!",
    "Test Steps": "1. Open Login page\n2. Enter email\n3. Enter password\n4. Click Log In",
    "Expected Result": "User is redirected to Dashboard; a session cookie is created",
    "Category": "Positive",
    "Priority": "P0",
    "Risk": "Medium"
  },
  {
    "Test Case ID": "TC-002",
    "Feature": "User Login",
    "Acceptance Criteria": "AC2",
    "Test Scenario": "Login fails with correct email but wrong password",
    "Preconditions": "Active, registered user; not locked",
    "Test Data": "Email: user@example.com\nPassword: WrongPass!",
    "Test Steps": "1. Open Login page\n2. Enter email\n3. Enter wrong password\n4. Click Log In",
    "Expected Result": "Inline error \"Invalid email 

In [43]:
import json
import re

def extract_json_array(text):
    """
    Extract a JSON array from an AI/model response.
    Handles plain JSON and JSON wrapped in Markdown code fences.
    """

    if text is None:
        raise ValueError("Model response is empty.")

    text = str(text).strip()

    # Remove Markdown code fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    # First try parsing the complete response
    try:
        data = json.loads(text)

        if isinstance(data, list):
            return data

        # Sometimes model returns {"test_cases": [...]}
        if isinstance(data, dict):
            for key in ["test_cases", "test_suite", "data", "results"]:
                if key in data and isinstance(data[key], list):
                    return data[key]

    except json.JSONDecodeError:
        pass

    # Find the first JSON array in the response
    start = text.find("[")
    end = text.rfind("]")

    if start != -1 and end != -1 and end > start:
        candidate = text[start:end + 1]

        try:
            data = json.loads(candidate)

            if isinstance(data, list):
                return data

        except json.JSONDecodeError as e:
            raise ValueError(
                f"Could not parse model response as JSON array: {e}"
            )

    raise ValueError(
        "No valid JSON array found in the model response."
    )

In [44]:
data_a = extract_json_array(structured_a_text)

df_a = pd.DataFrame(data_a)

for col in TEST_COLUMNS:
    if col not in df_a.columns:
        df_a[col] = ""

df_a["Category"] = df_a["Category"].apply(normalize_category)
df_a["Priority"] = df_a["Priority"].apply(normalize_priority)
df_a["Risk"] = df_a["Risk"].apply(normalize_risk)

print("Number of test cases:", len(df_a))
display(df_a)

Number of test cases: 9


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk
0,TC-001,User Login,AC1,Successful login with valid credentials,"Active, registered user; not logged in",Email: user@example.com\nPassword: CorrectPass...,1. Open Login page\n2. Enter email\n3. Enter p...,User is redirected to Dashboard; a session coo...,Positive,P0,Medium
1,TC-002,User Login,AC2,Login fails with correct email but wrong password,"Active, registered user; not locked",Email: user@example.com\nPassword: WrongPass!,1. Open Login page\n2. Enter email\n3. Enter w...,"Inline error ""Invalid email or password"" shown...",Negative,P0,Medium
2,TC-003,User Login,AC3,Login attempt with unregistered email,No account exists for the provided email,Email: unknown@example.com\nPassword: AnyPass123!,1. Open Login page\n2. Enter unregistered emai...,"Generic error ""Invalid email or password"" disp...",Negative,P1,Medium
3,TC-004,User Login,AC4,Login attempt with both fields blank,User is on Login page,Email: (blank)\nPassword: (blank),1. Open Login page\n2. Leave email and passwor...,Inline validation prompts to fill required fie...,Boundary,P1,Low
4,TC-005,User Login,AC5,Login attempt with invalid email format,User is on Login page,Email: invalid-email\nPassword: AnyPass123!,1. Open Login page\n2. Enter invalid email for...,"Inline message ""Enter a valid email address"" s...",Edge,P2,Low
5,TC-006,User Login,AC6,Account lock after 5 consecutive failed attemp...,"Active, registered user; account not currently...",Email: user@example.com\nPassword: WrongPass! ...,1. Open Login page\n2. Perform 5 login attempt...,"After 5th failed attempt, message ""Your accoun...",Negative,P0,High
6,TC-007,User Login,AC7,Email case‑insensitivity verification,"Active, registered user; not logged in",Email: USER@EXAMPLE.COM\nPassword: CorrectPass...,1. Open Login page\n2. Enter email in differen...,Login succeeds and user is redirected to Dashb...,Positive,P1,Medium
7,TC-008,User Login,AC8,Session persistence after browser refresh,User has successfully logged in and session is...,N/A,1. Perform successful login (see TC-001)\n2. R...,User remains on Dashboard; session cookie pers...,Positive,P1,Low
8,TC-009,User Login,AC9,Login attempt with deactivated account,User account is deactivated in the system,Email: deactivated@example.com\nPassword: AnyP...,1. Open Login page\n2. Enter deactivated accou...,"Message ""This account is inactive. Contact sup...",Negative,P1,Medium


## Positive / Negative / Boundary / Edge Test Cases

In [45]:
for category in ["Positive", "Negative", "Boundary", "Edge"]:
    print(f"\n===== {category} TEST CASES =====")
    display(df_a[df_a["Category"] == category])



===== Positive TEST CASES =====


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk
0,TC-001,User Login,AC1,Successful login with valid credentials,"Active, registered user; not logged in",Email: user@example.com\nPassword: CorrectPass...,1. Open Login page\n2. Enter email\n3. Enter p...,User is redirected to Dashboard; a session coo...,Positive,P0,Medium
6,TC-007,User Login,AC7,Email case‑insensitivity verification,"Active, registered user; not logged in",Email: USER@EXAMPLE.COM\nPassword: CorrectPass...,1. Open Login page\n2. Enter email in differen...,Login succeeds and user is redirected to Dashb...,Positive,P1,Medium
7,TC-008,User Login,AC8,Session persistence after browser refresh,User has successfully logged in and session is...,N/A,1. Perform successful login (see TC-001)\n2. R...,User remains on Dashboard; session cookie pers...,Positive,P1,Low



===== Negative TEST CASES =====


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk
1,TC-002,User Login,AC2,Login fails with correct email but wrong password,"Active, registered user; not locked",Email: user@example.com\nPassword: WrongPass!,1. Open Login page\n2. Enter email\n3. Enter w...,"Inline error ""Invalid email or password"" shown...",Negative,P0,Medium
2,TC-003,User Login,AC3,Login attempt with unregistered email,No account exists for the provided email,Email: unknown@example.com\nPassword: AnyPass123!,1. Open Login page\n2. Enter unregistered emai...,"Generic error ""Invalid email or password"" disp...",Negative,P1,Medium
5,TC-006,User Login,AC6,Account lock after 5 consecutive failed attemp...,"Active, registered user; account not currently...",Email: user@example.com\nPassword: WrongPass! ...,1. Open Login page\n2. Perform 5 login attempt...,"After 5th failed attempt, message ""Your accoun...",Negative,P0,High
8,TC-009,User Login,AC9,Login attempt with deactivated account,User account is deactivated in the system,Email: deactivated@example.com\nPassword: AnyP...,1. Open Login page\n2. Enter deactivated accou...,"Message ""This account is inactive. Contact sup...",Negative,P1,Medium



===== Boundary TEST CASES =====


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk
3,TC-004,User Login,AC4,Login attempt with both fields blank,User is on Login page,Email: (blank)\nPassword: (blank),1. Open Login page\n2. Leave email and passwor...,Inline validation prompts to fill required fie...,Boundary,P1,Low



===== Edge TEST CASES =====


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk
4,TC-005,User Login,AC5,Login attempt with invalid email format,User is on Login page,Email: invalid-email\nPassword: AnyPass123!,1. Open Login page\n2. Enter invalid email for...,"Inline message ""Enter a valid email address"" s...",Edge,P2,Low


## Feature Test Execution

In [46]:
# No real application/API/browser SUT is supplied in the capstone.
# This uses the existing deterministic QA execution gate and does not fabricate functional results.
execution_a = deterministic_qa_check(df_a, "User Login")
display(execution_a)


,Feature,Check,Status,Details
0,User Login,Required Columns,PASS,All required columns present
1,User Login,Empty Required Fields,PASS,No empty required fields
2,User Login,Duplicate Test Case IDs,PASS,0 duplicate rows
3,User Login,Valid Test Categories,PASS,All categories valid
4,User Login,Valid Priorities,PASS,All priorities valid
5,User Login,Valid Risks,PASS,All risks valid


## Feature Assertions / Validation

In [47]:
checks_a = run_deterministic_checks(df_a, EXPECTED_AC_A)
qa_checks_a = checks_to_dataframe(checks_a)
display(qa_checks_a)


,Check,Status,Details
0,Required Columns,FAIL,"{'status': 'PASS', 'missing_columns': [], 'ava..."
1,Duplicate IDs,PASS,"{'PASS': True, 'Duplicate IDs': []}"
2,Categories,PASS,"{'PASS': True, 'Invalid Categories': []}"
3,Priorities,PASS,"{'PASS': True, 'Invalid Priorities': []}"
4,Risks,PASS,"{'PASS': True, 'Invalid Risks': []}"
5,Blank Values,PASS,"{'PASS': True, 'Blank Fields': {}}"
6,AC Format,PASS,"{'PASS': True, 'Invalid AC References': []}"
7,AC Coverage,PASS,"{'Total ACs': 9, 'Covered ACs': 9, 'Missing AC..."
8,Category Coverage,PASS,"{'PASS': True, 'Missing Categories': []}"
9,Test Count,FAIL,"{'Count': 9, 'Minimum Expected': 10, 'PASS': F..."


## Actual Results / Expected vs Actual / Pass-Fail

In [48]:
def expected_actual_from_gate(df, execution_df):
    failed = (
        "Status" in execution_df.columns
        and (execution_df["Status"].astype(str).str.upper() == "FAIL").any()
    )
    gate_status = "FAIL" if failed else "PASS"
    actual_text = (
        "Deterministic test-suite execution gate passed"
        if gate_status == "PASS"
        else "Deterministic test-suite execution gate failed"
    )
    return pd.DataFrame([
        {
            "Test Case ID": row["Test Case ID"],
            "Acceptance Criteria": row["Acceptance Criteria"],
            "Expected Result": row["Expected Result"],
            "Actual Result": actual_text,
            "Pass / Fail": gate_status
        }
        for _, row in df.iterrows()
    ])

a_execution_results = expected_actual_from_gate(df_a, execution_a)
display(a_execution_results)


,Test Case ID,Acceptance Criteria,Expected Result,Actual Result,Pass / Fail
0,TC-001,AC1,User is redirected to Dashboard; a session coo...,Deterministic test-suite execution gate passed,PASS
1,TC-002,AC2,"Inline error ""Invalid email or password"" shown...",Deterministic test-suite execution gate passed,PASS
2,TC-003,AC3,"Generic error ""Invalid email or password"" disp...",Deterministic test-suite execution gate passed,PASS
3,TC-004,AC4,Inline validation prompts to fill required fie...,Deterministic test-suite execution gate passed,PASS
4,TC-005,AC5,"Inline message ""Enter a valid email address"" s...",Deterministic test-suite execution gate passed,PASS
5,TC-006,AC6,"After 5th failed attempt, message ""Your accoun...",Deterministic test-suite execution gate passed,PASS
6,TC-007,AC7,Login succeeds and user is redirected to Dashb...,Deterministic test-suite execution gate passed,PASS
7,TC-008,AC8,User remains on Dashboard; session cookie pers...,Deterministic test-suite execution gate passed,PASS
8,TC-009,AC9,"Message ""This account is inactive. Contact sup...",Deterministic test-suite execution gate passed,PASS


## Acceptance Criteria Mapping / Coverage

In [49]:
coverage_a = create_coverage_report(df_a, EXPECTED_AC_A, "User Login")
a_coverage_gaps = coverage_a[coverage_a["Coverage Status"].str.upper() == "MISSING"].copy()
display(coverage_a)
print("Coverage gaps:", len(a_coverage_gaps))


,Feature,Acceptance Criteria,Coverage Status,Test Case Count,Test Case IDs
0,User Login,AC1,COVERED,1,TC-001
1,User Login,AC2,COVERED,1,TC-002
2,User Login,AC3,COVERED,1,TC-003
3,User Login,AC4,COVERED,1,TC-004
4,User Login,AC5,COVERED,1,TC-005
5,User Login,AC6,COVERED,1,TC-006
6,User Login,AC7,COVERED,1,TC-007
7,User Login,AC8,COVERED,1,TC-008
8,User Login,AC9,COVERED,1,TC-009


Coverage gaps: 0


## Feature Reports / Logs / Complete Execution Output

In [50]:
qa_checks_a.to_csv("Feature_A_Deterministic_QA_Checks.csv", index=False, encoding="utf-8-sig")
coverage_a.to_csv("Feature_A_Coverage_Report.csv", index=False, encoding="utf-8-sig")
a_coverage_gaps.to_csv("Feature_A_Coverage_Gaps.csv", index=False, encoding="utf-8-sig")
a_execution_results.to_csv("Feature_A_Execution_Results.csv", index=False, encoding="utf-8-sig")
with open("Feature_A_Execution_Log.txt", "w", encoding="utf-8") as f:
    f.write("Feature: User Login\n")
    f.write("Execution mode: existing deterministic QA gate\n")
    f.write(f"Test cases: {len(df_a)}\n")
    f.write(f"Coverage gaps: {len(a_coverage_gaps)}\n")
print("Feature A reports and execution log created.")


Feature A reports and execution log created.


## Feature Final Summary

In [51]:
feature_a_summary = pd.DataFrame([{
    "Feature": "User Login",
    "Acceptance Criteria": len(EXPECTED_AC_A),
    "Covered ACs": int((coverage_a["Coverage Status"] == "COVERED").sum()),
    "Coverage Gaps": len(a_coverage_gaps),
    "Test Cases": len(df_a),
    "Execution Gate": "PASS" if not (execution_a["Status"].astype(str).str.upper() == "FAIL").any() else "FAIL",
    "Deterministic QA": "PASS" if all(r.get("PASS", False) for r in checks_a.values()) else "FAIL"
}])
display(feature_a_summary)


,Feature,Acceptance Criteria,Covered ACs,Coverage Gaps,Test Cases,Execution Gate,Deterministic QA
0,User Login,9,9,0,9,PASS,FAIL


# FEATURE B — Apply Promo Code at Checkout

**Mandatory execution order:** Feature B is executed completely through its final summary before the next feature begins.


## Feature Requirement

In [52]:
feature_b_requirement = 'feature_b = """\nFeature: Apply Promo Code at Checkout\n\nUser Story:\nAs a shopper, I want to apply a promo code at checkout so that I\nreceive a discount on my order.\n\nDescription:\nAt checkout there is a Promo code input and an Apply button.\nWhen a valid code is applied, the discount is reflected in the\norder summary and the total updates. An invalid or ineligible code\nshows an error and the total is unchanged.\n\nPromo code rules:\n\n- Percentage codes, e.g. SAVE10 = 10% off the item subtotal.\n- Fixed amount codes, e.g. FLAT200 = ₹200 off the item subtotal.\n- Codes are case-insensitive.\n- Some codes require a minimum subtotal.\n- Expired codes are rejected.\n- A single-use code cannot be reused by the same customer.\n- Only one code may be applied per order.\n- Applying a second code replaces the first only after confirmation.\n- Fixed discount cannot make subtotal negative.\n- Shipping and taxes are calculated on the discounted subtotal.'
print(feature_b_requirement)


feature_b = """
Feature: Apply Promo Code at Checkout

User Story:
As a shopper, I want to apply a promo code at checkout so that I
receive a discount on my order.

Description:
At checkout there is a Promo code input and an Apply button.
When a valid code is applied, the discount is reflected in the
order summary and the total updates. An invalid or ineligible code
shows an error and the total is unchanged.

Promo code rules:

- Percentage codes, e.g. SAVE10 = 10% off the item subtotal.
- Fixed amount codes, e.g. FLAT200 = ₹200 off the item subtotal.
- Codes are case-insensitive.
- Some codes require a minimum subtotal.
- Expired codes are rejected.
- A single-use code cannot be reused by the same customer.
- Only one code may be applied per order.
- Applying a second code replaces the first only after confirmation.
- Fixed discount cannot make subtotal negative.
- Shipping and taxes are calculated on the discounted subtotal.


## Acceptance Criteria — Loaded from External Editable File

In [53]:
feature_b_ac = load_acceptance_criteria("feature_b_acceptance_criteria.json", "Apply Promo Code at Checkout")
EXPECTED_AC_B = [item["id"] for item in feature_b_ac]
display(pd.DataFrame(feature_b_ac))
print("Expected AC IDs:", EXPECTED_AC_B)


Loaded 12 Acceptance Criteria for Apply Promo Code at Checkout.


,id,title,text
0,AC1,Valid percentage code.,Applying SAVE10 to a ₹1000 subtotal reduces it...
1,AC2,Valid fixed code above minimum.,Applying FLAT200 to a ₹1500 subtotal reduces i...
2,AC3,Fixed code below minimum.,Applying FLAT200 to an ₹800 subtotal is reject...
3,AC4,Expired code.,"Applying an expired code shows ""This code has ..."
4,AC5,Invalid code.,"Applying a non-existent code shows ""Invalid pr..."
5,AC6,Case insensitivity.,save10 behaves identically to SAVE10.
6,AC7,Already used.,Reapplying a single-use code already redeemed ...
7,AC8,Discount cap.,Applying FLAT200 to a ₹150 subtotal results in...
8,AC9,Replace existing code.,Applying a second code prompts the user to rep...
9,AC10,Empty input.,"Clicking Apply with no code shows ""Enter a pro..."


Expected AC IDs: ['AC1', 'AC2', 'AC3', 'AC4', 'AC5', 'AC6', 'AC7', 'AC8', 'AC9', 'AC10', 'AC11', 'AC12']


## Feature Data / Input Preparation

In [54]:
feature_b_requirement_context = build_requirement_context(feature_b_requirement, feature_b_ac)
print(feature_b_requirement_context)


feature_b = """
Feature: Apply Promo Code at Checkout

User Story:
As a shopper, I want to apply a promo code at checkout so that I
receive a discount on my order.

Description:
At checkout there is a Promo code input and an Apply button.
When a valid code is applied, the discount is reflected in the
order summary and the total updates. An invalid or ineligible code
shows an error and the total is unchanged.

Promo code rules:

- Percentage codes, e.g. SAVE10 = 10% off the item subtotal.
- Fixed amount codes, e.g. FLAT200 = ₹200 off the item subtotal.
- Codes are case-insensitive.
- Some codes require a minimum subtotal.
- Expired codes are rejected.
- A single-use code cannot be reused by the same customer.
- Only one code may be applied per order.
- Applying a second code replaces the first only after confirmation.
- Fixed discount cannot make subtotal negative.
- Shipping and taxes are calculated on the discounted subtotal.

Acceptance Criteria (loaded from external editable JSON):


## Feature Agent Prompt / Invocation / Scenario Generation

In [55]:
draft_b = generate_test_cases(feature_b_requirement_context)
print(draft_b)


**Draft Test Suite – “Apply Promo Code at Checkout”**  
*(All test cases are traceable to the Acceptance Criteria (AC) listed in the requirement. No functionality beyond the specification has been added.)*  

| # | Test Case ID | Acceptance Criteria ID | Test Scenario | Preconditions | Test Data (Cart subtotal, Promo code, etc.) | Test Steps | Expected Result | Category | Priority | Risk |
|---|--------------|------------------------|---------------|---------------|--------------------------------------------|------------|-----------------|----------|----------|------|
| 1 | TC‑001 | AC1 | Apply a valid percentage‑off code (SAVE10) to a subtotal that is well above the minimum | User is logged in, cart contains items totalling **₹1 000** | Subtotal = **₹1 000**; Promo code = **SAVE10** | 1. Navigate to Checkout  <br>2. Verify subtotal = ₹1 000 <br>3. Enter “SAVE10” in Promo field <br>4. Click **Apply** | Discount = 10 % of ₹1 000 = **₹100**; New subtotal = **₹900**; Order total (includi

## Feature Test Case Generation

In [56]:
print("Generated draft test suite for Apply Promo Code at Checkout.")
print(draft_b)


Generated draft test suite for Apply Promo Code at Checkout.
**Draft Test Suite – “Apply Promo Code at Checkout”**  
*(All test cases are traceable to the Acceptance Criteria (AC) listed in the requirement. No functionality beyond the specification has been added.)*  

| # | Test Case ID | Acceptance Criteria ID | Test Scenario | Preconditions | Test Data (Cart subtotal, Promo code, etc.) | Test Steps | Expected Result | Category | Priority | Risk |
|---|--------------|------------------------|---------------|---------------|--------------------------------------------|------------|-----------------|----------|----------|------|
| 1 | TC‑001 | AC1 | Apply a valid percentage‑off code (SAVE10) to a subtotal that is well above the minimum | User is logged in, cart contains items totalling **₹1 000** | Subtotal = **₹1 000**; Promo code = **SAVE10** | 1. Navigate to Checkout  <br>2. Verify subtotal = ₹1 000 <br>3. Enter “SAVE10” in Promo field <br>4. Click **Apply** | Discount = 10 % of ₹1 

## Feature Critic Agent

In [57]:
critique_b = critique_test_cases(feature_b_requirement_context, draft_b)
print(critique_b)


**QA Critique – “Apply Promo Code at Checkout” Draft Suite**

---

## 1. Coverage Report (per Acceptance Criterion)

| AC | Covered by Test(s) | Comments / Missing Scenarios |
|----|--------------------|------------------------------|
| **AC1** – SAVE10 on ₹1 000 | TC‑001, TC‑002 (boundary whole‑number) | ✔︎ basic discount & whole‑number boundary.  <br>‑ Missing verification that **shipping & tax are recomputed** on the discounted subtotal. |
| **AC2** – FLAT200 on ₹1 500 | TC‑010 | ✔︎ basic fixed‑amount discount.  <br>‑ Same missing shipping/tax verification. |
| **AC3** – Minimum‑order rule | TC‑008 (just below), TC‑009 (exact) | ✔︎ rejection & acceptance at the limit.  <br>‑ No test where **subtotal > min before applying but falls below after discount** (covered by AC12, not AC3). |
| **AC4** – Expired code | TC‑007 | ✔︎ message & no discount. |
| **AC5** – Non‑existent code | TC‑006 | ✔︎ message & no discount. |
| **AC6** – Case‑insensitivity | TC‑003 | ✔︎ lower‑case works. |
| **A

## Feature Improver Agent

In [58]:
final_b = improve_test_cases(feature_b_requirement_context, draft_b, critique_b)
print(final_b)


⏳ Groq TPM pacing: waiting 43.9s before the next AI call...
**Apply Promo Code at Checkout – Complete Traceable Test Suite**

| # | Test Case ID | Acceptance Criteria ID | Test Scenario | Preconditions | Test Data (Cart subtotal, Promo code, etc.) | Test Steps | Expected Result | Category | Priority | Risk |
|---|--------------|------------------------|---------------|---------------|--------------------------------------------|------------|-----------------|----------|----------|------|
| 1 | TC‑001 | AC1 | Apply a valid percentage‑off code (SAVE10) to a ₹1 000 subtotal | User logged in; cart subtotal = ₹1 000 | Subtotal = ₹1 000; Promo = SAVE10 | 1. Go to Checkout  <br>2. Verify displayed subtotal = ₹1 000  <br>3. Enter **SAVE10**  <br>4. Click **Apply** | Discount = 10 % × ₹1 000 = ₹100 → New subtotal = ₹900; Order total (incl. shipping & tax) reflects ₹900 subtotal | Positive | High | High (core discount) |
| 2 | TC‑002 | AC1 | Boundary – whole‑number discount on ₹1 200 subtotal | 

## Feature Validator Agent

In [59]:
validation_b = validate_test_suite(feature_b_requirement_context, final_b)
print(validation_b)


⏳ Groq TPM pacing: waiting 3.4s before the next AI call...
**FINAL COVERAGE STATUS**

| Acceptance Criteria ID | Covered by Test Cases? | Comments |
|------------------------|------------------------|----------|
| **AC1** – SAVE10 on ₹1 000 reduces subtotal by ₹100 | **PASS** | Covered by **TC‑001** (primary) and reinforced by **TC‑002** (boundary). |
| **AC2** – FLAT200 on ₹1 500 reduces subtotal by ₹200 | **FAIL** | No test case exercising a fixed‑amount discount on a qualifying subtotal. |
| **AC3** – FLAT200 on ₹800 rejected (minimum order) | **FAIL** | No test case for minimum‑subtotal validation. |
| **AC4** – Expired code shows “This code has expired.” | **FAIL** | No test case for expired‑code handling. |
| **AC5** – Non‑existent code shows “Invalid promo code.” | **FAIL** | No test case for unknown‑code handling. |
| **AC6** – Case‑insensitivity (save10 = SAVE10) | **PASS** | Covered by **TC‑003**. |
| **AC7** – Single‑use code already redeemed shows “This code has already bee

## Feature Final Test Case Structuring

In [60]:
structured_b_text = structure_test_cases(feature_b_requirement_context, final_b, "Apply Promo Code at Checkout")
print(structured_b_text[:3000])


[
  {
    "Test Case ID": "TC-001",
    "Feature": "Apply Promo Code at Checkout",
    "Acceptance Criteria": "AC1",
    "Test Scenario": "Apply a valid percentage‑off code (SAVE10) to a ₹1 000 subtotal",
    "Preconditions": "User logged in; cart subtotal = ₹1 000",
    "Test Data": "Subtotal = ₹1 000; Promo = SAVE10",
    "Test Steps": "1. Go to Checkout\n2. Verify displayed subtotal = ₹1 000\n3. Enter SAVE10\n4. Click Apply",
    "Expected Result": "Discount = 10 % × ₹1 000 = ₹100 → New subtotal = ₹900; Order total (incl. shipping & tax) reflects ₹900 subtotal",
    "Category": "Positive",
    "Priority": "P0",
    "Risk": "High"
  },
  {
    "Test Case ID": "TC-002",
    "Feature": "Apply Promo Code at Checkout",
    "Acceptance Criteria": "AC1",
    "Test Scenario": "Boundary – whole‑number discount on ₹1 200 subtotal",
    "Preconditions": "User logged in; cart subtotal = ₹1 000",
    "Test Data": "Subtotal = ₹1 200; Promo = SAVE10",
    "Test Steps": "1. Go to Checkout\n2. Verif

In [61]:
data_b = extract_json_array(structured_b_text)
df_b = pd.DataFrame(data_b)
for col in TEST_COLUMNS:
    if col not in df_b.columns:
        df_b[col] = ""
df_b["Category"] = df_b["Category"].apply(normalize_category)
df_b["Priority"] = df_b["Priority"].apply(normalize_priority)
df_b["Risk"] = df_b["Risk"].apply(normalize_risk)
print("Number of test cases:", len(df_b))
display(df_b)


Number of test cases: 4


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk
0,TC-001,Apply Promo Code at Checkout,AC1,Apply a valid percentage‑off code (SAVE10) to ...,User logged in; cart subtotal = ₹1 000,Subtotal = ₹1 000; Promo = SAVE10,1. Go to Checkout\n2. Verify displayed subtota...,Discount = 10 % × ₹1 000 = ₹100 → New subtotal...,Positive,P0,High
1,TC-002,Apply Promo Code at Checkout,AC1,Boundary – whole‑number discount on ₹1 200 sub...,User logged in; cart subtotal = ₹1 000,Subtotal = ₹1 200; Promo = SAVE10,1. Go to Checkout\n2. Verify displayed subtota...,Discount = ₹120 → New subtotal = ₹1 080; Total...,Boundary,P1,Medium
2,TC-003,Apply Promo Code at Checkout,AC6,Verify case‑insensitivity for percentage code,User logged in; cart subtotal = ₹1 000,Subtotal = ₹500; Promo = save10,1. Go to Checkout\n2. Verify displayed subtota...,Discount = ₹50 → New subtotal = ₹450; Total up...,Positive,P0,High
3,TC-004,Apply Promo Code at Checkout,AC11,Leading/trailing spaces are trimmed before val...,User logged in; cart subtotal = ₹1 000,Subtotal = ₹800; Promo = SAVE10,1. Go to Checkout\n2. Verify displayed subtota...,"Discount applied as SAVE10 → ₹80 discount, new...",Positive,P0,High


## Positive / Negative / Boundary / Edge Test Cases

In [62]:
for category in ["Positive", "Negative", "Boundary", "Edge"]:
    print(f"\n===== {category} TEST CASES =====")
    display(df_b[df_b["Category"] == category])



===== Positive TEST CASES =====


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk
0,TC-001,Apply Promo Code at Checkout,AC1,Apply a valid percentage‑off code (SAVE10) to ...,User logged in; cart subtotal = ₹1 000,Subtotal = ₹1 000; Promo = SAVE10,1. Go to Checkout\n2. Verify displayed subtota...,Discount = 10 % × ₹1 000 = ₹100 → New subtotal...,Positive,P0,High
2,TC-003,Apply Promo Code at Checkout,AC6,Verify case‑insensitivity for percentage code,User logged in; cart subtotal = ₹1 000,Subtotal = ₹500; Promo = save10,1. Go to Checkout\n2. Verify displayed subtota...,Discount = ₹50 → New subtotal = ₹450; Total up...,Positive,P0,High
3,TC-004,Apply Promo Code at Checkout,AC11,Leading/trailing spaces are trimmed before val...,User logged in; cart subtotal = ₹1 000,Subtotal = ₹800; Promo = SAVE10,1. Go to Checkout\n2. Verify displayed subtota...,"Discount applied as SAVE10 → ₹80 discount, new...",Positive,P0,High



===== Negative TEST CASES =====


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk



===== Boundary TEST CASES =====


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk
1,TC-002,Apply Promo Code at Checkout,AC1,Boundary – whole‑number discount on ₹1 200 sub...,User logged in; cart subtotal = ₹1 000,Subtotal = ₹1 200; Promo = SAVE10,1. Go to Checkout\n2. Verify displayed subtota...,Discount = ₹120 → New subtotal = ₹1 080; Total...,Boundary,P1,Medium



===== Edge TEST CASES =====


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk


## Feature Test Execution

In [63]:
# No real application/API/browser SUT is supplied in the capstone.
# This uses the existing deterministic QA execution gate and does not fabricate functional results.
execution_b = deterministic_qa_check(df_b, "Apply Promo Code at Checkout")
display(execution_b)


,Feature,Check,Status,Details
0,Apply Promo Code at Checkout,Required Columns,PASS,All required columns present
1,Apply Promo Code at Checkout,Empty Required Fields,PASS,No empty required fields
2,Apply Promo Code at Checkout,Duplicate Test Case IDs,PASS,0 duplicate rows
3,Apply Promo Code at Checkout,Valid Test Categories,PASS,All categories valid
4,Apply Promo Code at Checkout,Valid Priorities,PASS,All priorities valid
5,Apply Promo Code at Checkout,Valid Risks,PASS,All risks valid


## Feature Assertions / Validation

In [64]:
checks_b = run_deterministic_checks(df_b, EXPECTED_AC_B)
qa_checks_b = checks_to_dataframe(checks_b)
display(qa_checks_b)


,Check,Status,Details
0,Required Columns,FAIL,"{'status': 'PASS', 'missing_columns': [], 'ava..."
1,Duplicate IDs,PASS,"{'PASS': True, 'Duplicate IDs': []}"
2,Categories,PASS,"{'PASS': True, 'Invalid Categories': []}"
3,Priorities,PASS,"{'PASS': True, 'Invalid Priorities': []}"
4,Risks,PASS,"{'PASS': True, 'Invalid Risks': []}"
5,Blank Values,PASS,"{'PASS': True, 'Blank Fields': {}}"
6,AC Format,PASS,"{'PASS': True, 'Invalid AC References': []}"
7,AC Coverage,FAIL,"{'Total ACs': 12, 'Covered ACs': 3, 'Missing A..."
8,Category Coverage,FAIL,"{'PASS': False, 'Missing Categories': ['Edge',..."
9,Test Count,FAIL,"{'Count': 4, 'Minimum Expected': 10, 'PASS': F..."


## Actual Results / Expected vs Actual / Pass-Fail

In [65]:
def expected_actual_from_gate(df, execution_df):
    failed = (
        "Status" in execution_df.columns
        and (execution_df["Status"].astype(str).str.upper() == "FAIL").any()
    )
    gate_status = "FAIL" if failed else "PASS"
    actual_text = (
        "Deterministic test-suite execution gate passed"
        if gate_status == "PASS"
        else "Deterministic test-suite execution gate failed"
    )
    return pd.DataFrame([
        {
            "Test Case ID": row["Test Case ID"],
            "Acceptance Criteria": row["Acceptance Criteria"],
            "Expected Result": row["Expected Result"],
            "Actual Result": actual_text,
            "Pass / Fail": gate_status
        }
        for _, row in df.iterrows()
    ])

b_execution_results = expected_actual_from_gate(df_b, execution_b)
display(b_execution_results)


,Test Case ID,Acceptance Criteria,Expected Result,Actual Result,Pass / Fail
0,TC-001,AC1,Discount = 10 % × ₹1 000 = ₹100 → New subtotal...,Deterministic test-suite execution gate passed,PASS
1,TC-002,AC1,Discount = ₹120 → New subtotal = ₹1 080; Total...,Deterministic test-suite execution gate passed,PASS
2,TC-003,AC6,Discount = ₹50 → New subtotal = ₹450; Total up...,Deterministic test-suite execution gate passed,PASS
3,TC-004,AC11,"Discount applied as SAVE10 → ₹80 discount, new...",Deterministic test-suite execution gate passed,PASS


## Acceptance Criteria Mapping / Coverage

In [66]:
coverage_b = create_coverage_report(df_b, EXPECTED_AC_B, "Apply Promo Code at Checkout")
b_coverage_gaps = coverage_b[coverage_b["Coverage Status"].str.upper() == "MISSING"].copy()
display(coverage_b)
print("Coverage gaps:", len(b_coverage_gaps))


,Feature,Acceptance Criteria,Coverage Status,Test Case Count,Test Case IDs
0,Apply Promo Code at Checkout,AC1,COVERED,2,"TC-001, TC-002"
1,Apply Promo Code at Checkout,AC2,MISSING,0,
2,Apply Promo Code at Checkout,AC3,MISSING,0,
3,Apply Promo Code at Checkout,AC4,MISSING,0,
4,Apply Promo Code at Checkout,AC5,MISSING,0,
5,Apply Promo Code at Checkout,AC6,COVERED,1,TC-003
6,Apply Promo Code at Checkout,AC7,MISSING,0,
7,Apply Promo Code at Checkout,AC8,MISSING,0,
8,Apply Promo Code at Checkout,AC9,MISSING,0,
9,Apply Promo Code at Checkout,AC10,MISSING,0,


Coverage gaps: 9


## Feature Reports / Logs / Complete Execution Output

In [67]:
qa_checks_b.to_csv("Feature_B_Deterministic_QA_Checks.csv", index=False, encoding="utf-8-sig")
coverage_b.to_csv("Feature_B_Coverage_Report.csv", index=False, encoding="utf-8-sig")
b_coverage_gaps.to_csv("Feature_B_Coverage_Gaps.csv", index=False, encoding="utf-8-sig")
b_execution_results.to_csv("Feature_B_Execution_Results.csv", index=False, encoding="utf-8-sig")
with open("Feature_B_Execution_Log.txt", "w", encoding="utf-8") as f:
    f.write("Feature: Apply Promo Code at Checkout\n")
    f.write("Execution mode: existing deterministic QA gate\n")
    f.write(f"Test cases: {len(df_b)}\n")
    f.write(f"Coverage gaps: {len(b_coverage_gaps)}\n")
print("Feature B reports and execution log created.")


Feature B reports and execution log created.


## Feature Final Summary

In [68]:
feature_b_summary = pd.DataFrame([{
    "Feature": "Apply Promo Code at Checkout",
    "Acceptance Criteria": len(EXPECTED_AC_B),
    "Covered ACs": int((coverage_b["Coverage Status"] == "COVERED").sum()),
    "Coverage Gaps": len(b_coverage_gaps),
    "Test Cases": len(df_b),
    "Execution Gate": "PASS" if not (execution_b["Status"].astype(str).str.upper() == "FAIL").any() else "FAIL",
    "Deterministic QA": "PASS" if all(r.get("PASS", False) for r in checks_b.values()) else "FAIL"
}])
display(feature_b_summary)


,Feature,Acceptance Criteria,Covered ACs,Coverage Gaps,Test Cases,Execution Gate,Deterministic QA
0,Apply Promo Code at Checkout,12,3,9,4,PASS,FAIL


# FINAL COMBINED REPORTING

Feature A and Feature B are both fully completed before this section begins.


## Combined Test Suite

In [69]:
final_test_suite = pd.concat([df_a, df_b], ignore_index=True)
display(final_test_suite)
print("TOTAL FINAL TEST CASES:", len(final_test_suite))


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk
0,TC-001,User Login,AC1,Successful login with valid credentials,"Active, registered user; not logged in",Email: user@example.com\nPassword: CorrectPass...,1. Open Login page\n2. Enter email\n3. Enter p...,User is redirected to Dashboard; a session coo...,Positive,P0,Medium
1,TC-002,User Login,AC2,Login fails with correct email but wrong password,"Active, registered user; not locked",Email: user@example.com\nPassword: WrongPass!,1. Open Login page\n2. Enter email\n3. Enter w...,"Inline error ""Invalid email or password"" shown...",Negative,P0,Medium
2,TC-003,User Login,AC3,Login attempt with unregistered email,No account exists for the provided email,Email: unknown@example.com\nPassword: AnyPass123!,1. Open Login page\n2. Enter unregistered emai...,"Generic error ""Invalid email or password"" disp...",Negative,P1,Medium
3,TC-004,User Login,AC4,Login attempt with both fields blank,User is on Login page,Email: (blank)\nPassword: (blank),1. Open Login page\n2. Leave email and passwor...,Inline validation prompts to fill required fie...,Boundary,P1,Low
4,TC-005,User Login,AC5,Login attempt with invalid email format,User is on Login page,Email: invalid-email\nPassword: AnyPass123!,1. Open Login page\n2. Enter invalid email for...,"Inline message ""Enter a valid email address"" s...",Edge,P2,Low
5,TC-006,User Login,AC6,Account lock after 5 consecutive failed attemp...,"Active, registered user; account not currently...",Email: user@example.com\nPassword: WrongPass! ...,1. Open Login page\n2. Perform 5 login attempt...,"After 5th failed attempt, message ""Your accoun...",Negative,P0,High
6,TC-007,User Login,AC7,Email case‑insensitivity verification,"Active, registered user; not logged in",Email: USER@EXAMPLE.COM\nPassword: CorrectPass...,1. Open Login page\n2. Enter email in differen...,Login succeeds and user is redirected to Dashb...,Positive,P1,Medium
7,TC-008,User Login,AC8,Session persistence after browser refresh,User has successfully logged in and session is...,N/A,1. Perform successful login (see TC-001)\n2. R...,User remains on Dashboard; session cookie pers...,Positive,P1,Low
8,TC-009,User Login,AC9,Login attempt with deactivated account,User account is deactivated in the system,Email: deactivated@example.com\nPassword: AnyP...,1. Open Login page\n2. Enter deactivated accou...,"Message ""This account is inactive. Contact sup...",Negative,P1,Medium
9,TC-001,Apply Promo Code at Checkout,AC1,Apply a valid percentage‑off code (SAVE10) to ...,User logged in; cart subtotal = ₹1 000,Subtotal = ₹1 000; Promo = SAVE10,1. Go to Checkout\n2. Verify displayed subtota...,Discount = 10 % × ₹1 000 = ₹100 → New subtotal...,Positive,P0,High


TOTAL FINAL TEST CASES: 13


## Combined Acceptance Criteria Coverage

In [70]:
coverage_report = pd.concat([coverage_a, coverage_b], ignore_index=True)
coverage_gaps = coverage_report[
    coverage_report["Coverage Status"].str.upper() == "MISSING"
].copy()
display(coverage_report)
print("Coverage gaps:", len(coverage_gaps))


,Feature,Acceptance Criteria,Coverage Status,Test Case Count,Test Case IDs
0,User Login,AC1,COVERED,1,TC-001
1,User Login,AC2,COVERED,1,TC-002
2,User Login,AC3,COVERED,1,TC-003
3,User Login,AC4,COVERED,1,TC-004
4,User Login,AC5,COVERED,1,TC-005
5,User Login,AC6,COVERED,1,TC-006
6,User Login,AC7,COVERED,1,TC-007
7,User Login,AC8,COVERED,1,TC-008
8,User Login,AC9,COVERED,1,TC-009
9,Apply Promo Code at Checkout,AC1,COVERED,2,"TC-001, TC-002"


Coverage gaps: 9


## Category / Priority / Risk Reporting

In [71]:
category_summary_a = (
    df_a.groupby("Category")
    .size()
    .reset_index(name="Test Case Count")
)

category_summary_a["Feature"] = "User Login"

category_summary_b = (
    df_b.groupby("Category")
    .size()
    .reset_index(name="Test Case Count")
)

category_summary_b["Feature"] = "Apply Promo Code at Checkout"

category_summary = pd.concat(
    [
        category_summary_a,
        category_summary_b
    ],
    ignore_index=True
)

category_summary = category_summary[
    ["Feature", "Category", "Test Case Count"]
]

display(category_summary)

,Feature,Category,Test Case Count
0,User Login,Boundary,1
1,User Login,Edge,1
2,User Login,Negative,4
3,User Login,Positive,3
4,Apply Promo Code at Checkout,Boundary,1
5,Apply Promo Code at Checkout,Positive,3


In [72]:
priority_summary = pd.concat(
    [
        df_a.assign(Feature="User Login"),
        df_b.assign(
            Feature="Apply Promo Code at Checkout"
        )
    ],
    ignore_index=True
)

priority_summary = (
    priority_summary
    .groupby(["Feature", "Priority"])
    .size()
    .reset_index(name="Test Case Count")
)

display(priority_summary)

,Feature,Priority,Test Case Count
0,Apply Promo Code at Checkout,P0,3
1,Apply Promo Code at Checkout,P1,1
2,User Login,P0,3
3,User Login,P1,5
4,User Login,P2,1


In [73]:
risk_summary = pd.concat(
    [
        df_a.assign(Feature="User Login"),
        df_b.assign(
            Feature="Apply Promo Code at Checkout"
        )
    ],
    ignore_index=True
)

risk_summary = (
    risk_summary
    .groupby(["Feature", "Risk"])
    .size()
    .reset_index(name="Test Case Count")
)

display(risk_summary)

,Feature,Risk,Test Case Count
0,Apply Promo Code at Checkout,High,3
1,Apply Promo Code at Checkout,Medium,1
2,User Login,High,1
3,User Login,Low,3
4,User Login,Medium,5


## Combined Expected vs Actual / Execution Results

In [74]:
execution_report = pd.concat([a_execution_results, b_execution_results], ignore_index=True)
display(execution_report)


,Test Case ID,Acceptance Criteria,Expected Result,Actual Result,Pass / Fail
0,TC-001,AC1,User is redirected to Dashboard; a session coo...,Deterministic test-suite execution gate passed,PASS
1,TC-002,AC2,"Inline error ""Invalid email or password"" shown...",Deterministic test-suite execution gate passed,PASS
2,TC-003,AC3,"Generic error ""Invalid email or password"" disp...",Deterministic test-suite execution gate passed,PASS
3,TC-004,AC4,Inline validation prompts to fill required fie...,Deterministic test-suite execution gate passed,PASS
4,TC-005,AC5,"Inline message ""Enter a valid email address"" s...",Deterministic test-suite execution gate passed,PASS
5,TC-006,AC6,"After 5th failed attempt, message ""Your accoun...",Deterministic test-suite execution gate passed,PASS
6,TC-007,AC7,Login succeeds and user is redirected to Dashb...,Deterministic test-suite execution gate passed,PASS
7,TC-008,AC8,User remains on Dashboard; session cookie pers...,Deterministic test-suite execution gate passed,PASS
8,TC-009,AC9,"Message ""This account is inactive. Contact sup...",Deterministic test-suite execution gate passed,PASS
9,TC-001,AC1,Discount = 10 % × ₹1 000 = ₹100 → New subtotal...,Deterministic test-suite execution gate passed,PASS


## Final Project Summary

In [75]:
project_summary = pd.DataFrame([{
    "Project": "Agentic AI Test Case Generator",
    "Model": MODEL,
    "Feature A": "User Login",
    "Feature B": "Apply Promo Code at Checkout",
    "Total Test Cases": len(final_test_suite),
    "Positive Cases": int((final_test_suite["Category"] == "Positive").sum()),
    "Negative Cases": int((final_test_suite["Category"] == "Negative").sum()),
    "Boundary Cases": int((final_test_suite["Category"] == "Boundary").sum()),
    "Edge Cases": int((final_test_suite["Category"] == "Edge").sum()),
    "Coverage Gaps": len(coverage_gaps),
    "Feature A Execution Gate": "PASS" if not (execution_a["Status"].astype(str).str.upper() == "FAIL").any() else "FAIL",
    "Feature B Execution Gate": "PASS" if not (execution_b["Status"].astype(str).str.upper() == "FAIL").any() else "FAIL"
}])
display(project_summary)


,Project,Model,Feature A,Feature B,Total Test Cases,Positive Cases,Negative Cases,Boundary Cases,Edge Cases,Coverage Gaps,Feature A Execution Gate,Feature B Execution Gate
0,Agentic AI Test Case Generator,openai/gpt-oss-120b,User Login,Apply Promo Code at Checkout,13,6,4,2,1,9,PASS,PASS


## Core CSV Outputs

In [76]:
final_test_suite.to_csv("final_test_suite.csv", index=False, encoding="utf-8-sig")
qa_checks_a.to_csv("validation_feature_a.csv", index=False, encoding="utf-8-sig")
qa_checks_b.to_csv("validation_feature_b.csv", index=False, encoding="utf-8-sig")
execution_report.to_csv("execution_report.csv", index=False, encoding="utf-8-sig")
coverage_report.to_csv("coverage_report.csv", index=False, encoding="utf-8-sig")
coverage_gaps.to_csv("coverage_gaps.csv", index=False, encoding="utf-8-sig")
project_summary.to_csv("final_summary.csv", index=False, encoding="utf-8-sig")
category_coverage = final_test_suite["Category"].value_counts().reset_index()
category_coverage.columns = ["Category", "Test Case Count"]
category_coverage.to_csv("category_coverage.csv", index=False, encoding="utf-8-sig")
print("Core CSV outputs created.")


Core CSV outputs created.


## Gherkin / BDD Outputs

In [77]:
gherkin_a = dataframe_to_gherkin(df_a, "User Login")
gherkin_b = dataframe_to_gherkin(df_b, "Apply Promo Code at Checkout")
with open("Feature_A_User_Login.feature", "w", encoding="utf-8") as f:
    f.write(gherkin_a)
with open("Feature_B_Promo_Code.feature", "w", encoding="utf-8") as f:
    f.write(gherkin_b)
print("Gherkin outputs created.")


Gherkin outputs created.


In [78]:
# ============================================================
# DESIGN WRITE-UP
# ============================================================

design_writeup = """
# Agentic AI Test Case Generator — Design Writeup

## 1. Project Overview

The Agentic AI Test Case Generator generates requirement-traceable QA
test suites from software requirements and Acceptance Criteria.

The project covers two features:

1. User Login
2. Apply Promo Code at Checkout

The generated test suite contains:

- Positive test cases
- Negative test cases
- Boundary test cases
- Edge test cases

Each test case is mapped to the relevant Acceptance Criterion to
maintain requirement traceability.

---

## 2. Project Objective

The objective of this project is to demonstrate how an Agentic AI
workflow can assist QA engineers in generating comprehensive test
cases from requirements.

The workflow uses multiple stages:

Generate → Critique → Improve → Structure → Validate → Execute →
Coverage Analysis → Reporting

This approach provides both AI-assisted test generation and
deterministic QA validation.

---

## 3. External Acceptance Criteria

The Acceptance Criteria are maintained as two external editable JSON
files:

- feature_a_acceptance_criteria.json
- feature_b_acceptance_criteria.json

The notebook loads these files dynamically during execution.

This allows Acceptance Criteria to be changed without modifying the
core agent implementation, prompts, generation logic, validation
logic, or reporting logic.

These two JSON files are the only external editable project inputs.

---

## 4. Common Project Setup

The notebook contains a common setup section used by both features.

The common section includes:

- Project title
- Project requirements
- Python imports
- Configuration
- Model/API configuration
- Authentication
- Agent initialization
- Tool initialization
- Common utility functions
- Prompt helpers
- JSON parsing utilities
- Validation utilities
- Reporting utilities

All common functionality is initialized before Feature A starts.

---

## 5. Feature Execution Order

The project uses a strict sequential execution flow.

### Feature A — User Login

Feature A completes its entire workflow before Feature B begins.

The Feature A workflow is:

Requirement
→ Acceptance Criteria Loading
→ Acceptance Criteria Display
→ Data/Input Preparation
→ Agent Prompt
→ Agent Invocation
→ Test Scenario Generation
→ Test Case Generation
→ Positive Test Cases
→ Negative Test Cases
→ Boundary Test Cases
→ Edge Cases
→ Test Case Display
→ Test Execution / QA Gate
→ Assertions / Validation
→ Actual Results
→ Expected vs Actual Comparison
→ Pass / Fail Status
→ Acceptance Criteria Mapping
→ Acceptance Criteria Coverage
→ Reports
→ Logs
→ Feature A Final Summary

Feature A is fully completed before Feature B starts.

### Feature B — Apply Promo Code at Checkout

Feature B then follows the same complete workflow:

Requirement
→ Acceptance Criteria Loading
→ Acceptance Criteria Display
→ Data/Input Preparation
→ Agent Prompt
→ Agent Invocation
→ Test Scenario Generation
→ Test Case Generation
→ Positive Test Cases
→ Negative Test Cases
→ Boundary Test Cases
→ Edge Cases
→ Test Case Display
→ Test Execution / QA Gate
→ Assertions / Validation
→ Actual Results
→ Expected vs Actual Comparison
→ Pass / Fail Status
→ Acceptance Criteria Mapping
→ Acceptance Criteria Coverage
→ Reports
→ Logs
→ Feature B Final Summary

Only after Feature B is complete does combined reporting begin.

---

## 6. Agentic Workflow

### Generator Agent

The Generator Agent creates the initial test suite from the supplied
requirement and Acceptance Criteria.

The prompt instructs the agent to consider:

- Requirement traceability
- Positive scenarios
- Negative scenarios
- Boundary scenarios
- Edge scenarios
- Business rules
- Validation messages
- Thresholds
- Timing conditions
- State changes
- Priorities
- Risks

The Generator provides a broad first-pass test suite.

### Critic Agent

The Critic Agent independently reviews the generated test suite.

The critic checks for:

- Missing Acceptance Criteria coverage
- Missing scenarios
- Missing positive cases
- Missing negative cases
- Missing boundary cases
- Missing edge cases
- Missing thresholds
- Missing timing conditions
- Missing state transitions
- Weak expected results
- Duplicate or overlapping scenarios
- Traceability issues
- Unsupported assumptions

The critic produces feedback identifying areas that require
improvement.

### Improver Agent

The Improver Agent uses the original requirement, Acceptance Criteria,
generated test suite, and critic feedback to improve the suite.

The Improver preserves the supplied business requirements and avoids
inventing unsupported functionality.

---

## 7. Test Case Structuring

The improved test cases are converted into a consistent structured
format.

The structured test suite contains information such as:

- Test Case ID
- Acceptance Criterion
- Scenario
- Preconditions
- Test Data
- Test Steps
- Expected Result
- Category
- Priority
- Risk

This structure allows the generated cases to be validated,
executed, analyzed, and exported consistently.

---

## 8. Deterministic Validation

Because AI-generated output can vary between executions, deterministic
Python validation is used as an independent quality gate.

The validation checks include:

- Required columns
- Test Case ID presence
- Duplicate Test Case IDs
- Acceptance Criteria format
- Acceptance Criteria traceability
- Scenario presence
- Test steps presence
- Expected result presence
- Valid test categories
- Valid priorities
- Valid risks
- Category coverage
- Acceptance Criteria coverage

The deterministic validation layer operates independently of the
LLM's reasoning.

This provides repeatable structural quality checks.

---

## 9. Test Categories

### Positive Test Cases

Verify valid inputs and expected successful business flows.

### Negative Test Cases

Verify invalid inputs, rejected operations, and error conditions.

### Boundary Test Cases

Verify values at, below, and above defined limits, thresholds,
minimums, maximums, timing limits, and other boundaries.

### Edge Test Cases

Verify unusual combinations, special conditions, state changes, and
less-common scenarios that may expose additional issues.

---

## 10. Test Execution

The current capstone project does not provide a real production
application, API, or browser-based System Under Test.

Therefore, the notebook does not fabricate functional application
PASS/FAIL results.

The execution stage performs a deterministic QA gate against the
generated test-suite artifacts.

The execution process verifies that generated test cases satisfy the
required structural and QA conditions.

The resulting execution reports identify whether the generated
artifacts pass the defined validation gate.

A real application, API, or browser System Under Test can be connected
to the execution stage in the future.

---

## 11. Expected vs Actual Results

Each generated test case contains an expected result derived from the
requirement and Acceptance Criteria.

For the current implementation, the actual execution result
represents the deterministic test-suite QA gate.

The notebook therefore does not claim that a real application
functionally passed or failed when no real System Under Test is
available.

This keeps the execution results transparent and prevents fabricated
functional results.

---

## 12. Acceptance Criteria Mapping

Each generated test case is linked to an Acceptance Criterion.

This provides requirement traceability and allows the project to
identify:

- Which Acceptance Criteria are covered
- Which test cases cover each Acceptance Criterion
- Which Acceptance Criteria have insufficient coverage
- Which Acceptance Criteria have no mapped test cases

---

## 13. Acceptance Criteria Coverage

Coverage analysis is performed after test-case generation,
improvement, structuring, and validation.

The coverage report provides a view of Acceptance Criteria against
their corresponding test cases.

Coverage gaps are separately identified for QA review.

This makes it possible to review the completeness of the generated
test suite before final submission.

---

## 14. Reporting

The notebook generates project reporting artifacts including:

- Final test suite
- Feature A validation report
- Feature B validation report
- Category coverage report
- Acceptance Criteria coverage report
- Coverage gap report
- Feature A execution report
- Feature B execution report
- Final summary
- Design write-up
- Reflection report

All reporting is generated from the same notebook execution.

---

## 15. Logs and Execution Evidence

The feature workflows retain execution information so that the
generation, validation, and reporting stages can be reviewed.

Feature A logging and reporting are completed before Feature B starts.

Feature B logging and reporting are completed before combined
reporting begins.

This preserves the required sequential feature execution.

---

## 16. Human Review

The Agentic AI system is intended to assist QA engineers rather than
replace human QA judgment.

Human review remains necessary for:

- Business intent
- Requirement ambiguity
- Risk interpretation
- Environment-specific test data
- Application-specific behavior
- Automation feasibility
- Unsupported assumptions
- Final test-suite suitability

AI-generated scenarios should therefore be reviewed before being
used as production test cases.

---

## 17. Genuine Value of the Agentic Approach

The primary value of the agentic approach is reducing repetitive
effort in test-scenario brainstorming and coverage review.

The Generate → Critique → Improve workflow allows the system to:

1. Generate a broad first-pass test suite.
2. Independently critique the generated suite.
3. Identify missing coverage and weaknesses.
4. Improve the test suite using the critique.
5. Apply deterministic validation.
6. Produce requirement traceability and coverage reports.

This combines AI-assisted generation with repeatable QA controls.

---

## 18. Limitations

The current implementation has the following limitations:

1. AI-generated test cases require human review.
2. LLM output may vary between executions.
3. The current project does not contain a real System Under Test.
4. Functional application PASS/FAIL results cannot be claimed without
   an actual application, API, or browser target.
5. Environment-specific behavior requires the target environment.
6. Requirements that are not supplied should not be assumed by the
   agent.
7. Deterministic validation verifies generated artifacts but does not
   replace functional testing of a real application.

---

## 19. Project Outputs

The notebook produces the final project outputs from the completed
Feature A and Feature B workflows.

Typical outputs include:

- final_test_suite.csv
- validation_feature_a.csv
- validation_feature_b.csv
- category_coverage.csv
- coverage_report.csv
- coverage_gaps.csv
- execution_feature_a.csv
- execution_feature_b.csv
- final_summary.csv
- DESIGN_WRITEUP.md
- Agentic_AI_Capstone_Reflection.pdf

These files are generated outputs.

The implementation remains inside the single project notebook.

The only external editable inputs are:

- feature_a_acceptance_criteria.json
- feature_b_acceptance_criteria.json

---

## 20. Conclusion

The Agentic AI Test Case Generator demonstrates an end-to-end QA
workflow that transforms requirements and Acceptance Criteria into
structured, categorized, requirement-traceable test suites.

The Generate → Critique → Improve process provides iterative AI
assistance, while deterministic validation provides repeatable
structural quality controls.

Feature A is completed fully before Feature B begins, and Feature B
is completed fully before combined reporting is performed.

The complete implementation, execution, validation, coverage analysis,
reporting, and output generation are maintained within the same
notebook.

The project therefore provides a structured demonstration of how
Agentic AI can support QA test design while retaining deterministic
validation and human review.
"""

# ------------------------------------------------------------
# Save Design Writeup
# ------------------------------------------------------------

design_writeup_file = "DESIGN_WRITEUP.md"

with open(design_writeup_file, "w", encoding="utf-8") as f:
    f.write(design_writeup)

print("✅ DESIGN_WRITEUP.md created successfully")
print(f"File: {design_writeup_file}")

✅ DESIGN_WRITEUP.md created successfully
File: DESIGN_WRITEUP.md


## Design Writeup

The existing project architecture is retained: Requirement → Generate → Critique → Improve → Validator → Structure → Deterministic QA → Coverage → Reporting.

The only structural change is execution order: Feature A's existing workflow is completed before Feature B starts. Acceptance Criteria are externalized to two editable JSON files so AC edits do not require modifying the agent implementation.

The supplied capstone has no real application/API/browser SUT, so the execution stage uses the existing deterministic QA gate and explicitly avoids claiming functional application results.


## Reflection PDF

In [80]:
# ============================================================

# AGENTIC AI CAPSTONE - REFLECTION PDF

# ============================================================

from reportlab.lib.pagesizes import A4

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle

from reportlab.lib import colors

from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

from reportlab.lib.enums import TA_CENTER

from reportlab.lib.units import mm

import os

import pandas as pd

pdf_file = "Agentic_AI_Capstone_Reflection.pdf"

styles = getSampleStyleSheet()

title_style = ParagraphStyle(

    "TitleStyle",

    parent=styles["Title"],

    fontSize=18,

    leading=22,

    alignment=TA_CENTER,

    spaceAfter=12

)

heading_style = ParagraphStyle(

    "HeadingStyle",

    parent=styles["Heading1"],

    fontSize=13,

    leading=16,

    spaceBefore=8,

    spaceAfter=5

)

body_style = ParagraphStyle(

    "BodyStyle",

    parent=styles["BodyText"],

    fontSize=9.5,

    leading=13,

    spaceAfter=6

)

# ------------------------------------------------------------

# SAFELY READ FINAL TEST SUITE

# ------------------------------------------------------------

if "final_test_suite" not in globals():

    raise RuntimeError(

        "final_test_suite is not available. "

        "Please run the Feature A and Feature B generation/validation cells first."

    )

if not isinstance(final_test_suite, pd.DataFrame):

    raise RuntimeError("final_test_suite is not a pandas DataFrame.")

# ------------------------------------------------------------

# CALCULATE SUMMARY VALUES

# ------------------------------------------------------------

total_cases = len(final_test_suite)

if "Category" in final_test_suite.columns:

    category_counts = (

        final_test_suite["Category"]

        .fillna("")

        .astype(str)

        .str.strip()

        .value_counts()

    )

    positive_cases = int(category_counts.get("Positive", 0))

    negative_cases = int(category_counts.get("Negative", 0))

    boundary_cases = int(category_counts.get("Boundary", 0))

    edge_cases = int(category_counts.get("Edge", 0))

else:

    positive_cases = 0

    negative_cases = 0

    boundary_cases = 0

    edge_cases = 0

# ------------------------------------------------------------

# COVERAGE GAP COUNT

# ------------------------------------------------------------

coverage_gap_count = 0

if "coverage_gaps" in globals():

    if isinstance(coverage_gaps, pd.DataFrame):

        coverage_gap_count = len(coverage_gaps)

# ------------------------------------------------------------

# FINAL STATUS

# ------------------------------------------------------------

final_status = "PASS"

if "validation_a" in globals() and isinstance(validation_a, pd.DataFrame):

    if "Status" in validation_a.columns:

        if (validation_a["Status"].astype(str).str.upper() == "FAIL").any():

            final_status = "FAIL"

if "validation_b" in globals() and isinstance(validation_b, pd.DataFrame):

    if "Status" in validation_b.columns:

        if (validation_b["Status"].astype(str).str.upper() == "FAIL").any():

            final_status = "FAIL"

# ------------------------------------------------------------

# CREATE PDF

# ------------------------------------------------------------

doc = SimpleDocTemplate(

    pdf_file,

    pagesize=A4,

    rightMargin=16 * mm,

    leftMargin=16 * mm,

    topMargin=16 * mm,

    bottomMargin=16 * mm

)

content = [

    Paragraph(

        "Agentic AI Test Case Generator",

        title_style

    ),

    Paragraph(

        "Short Reflection",

        heading_style

    ),

    Paragraph(

        "The agent added value through broad first-pass test scenario generation "

        "and independent critique. Instead of manually brainstorming every "

        "scenario, the generator created a structured starting point and the "

        "critic reviewed coverage, thresholds, timing, state changes, and "

        "requirement traceability. The improver then used that critique to "

        "produce the final test suite.",

        body_style

    ),

    Paragraph(

        "Compared with writing every case manually, the main time saving is "

        "in repetitive scenario ideation and cross-checking. The agent does "

        "not replace QA judgment. Human review is still required to verify "

        "business intent and reject unsupported assumptions. Deterministic "

        "Python validation is therefore used as a separate quality gate.",

        body_style

    ),

    Paragraph(

        "Observed Final Outputs",

        heading_style

    ),

    Table(

        [

            ["Measure", "Value"],

            ["Total test cases", str(total_cases)],

            ["Positive", str(positive_cases)],

            ["Negative", str(negative_cases)],

            ["Boundary", str(boundary_cases)],

            ["Edge", str(edge_cases)],

            ["Coverage gaps", str(coverage_gap_count)],

            ["Final status", final_status]

        ],

        colWidths=[80 * mm, 70 * mm],

        style=TableStyle([

            ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),

            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),

            ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),

            ("FONTSIZE", (0, 0), (-1, -1), 9),

            ("VALIGN", (0, 0), (-1, -1), "TOP"),

            ("PADDING", (0, 0), (-1, -1), 5)

        ])

    ),

    Spacer(1, 8),

    Paragraph(

        "What Still Requires Manual Review",

        heading_style

    ),

    Paragraph(

        "Human review remains necessary for business-risk interpretation, "

        "environment-specific test data, automation feasibility, and "

        "requirement ambiguity. The agent should therefore be treated as "

        "a test-design accelerator and reviewer rather than the final "

        "authority.",

        body_style

    )

]

doc.build(content)

print("=" * 60)

print("REFLECTION PDF")

print("=" * 60)

print("PDF created successfully:", pdf_file)

print("File exists:", os.path.exists(pdf_file))

print("File size:", os.path.getsize(pdf_file), "bytes")


REFLECTION PDF
PDF created successfully: Agentic_AI_Capstone_Reflection.pdf
File exists: True
File size: 2792 bytes


In [81]:
# ============================================================
# FINAL SUBMISSION ARTIFACTS
# ============================================================

import os
import pandas as pd

print("=" * 60)
print("CREATING FINAL SUBMISSION ARTIFACTS")
print("=" * 60)

# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def save_dataframe_if_available(variable_name, file_name):
    """
    Save a dataframe only when the variable exists and is a DataFrame.
    """
    if variable_name in globals():
        obj = globals()[variable_name]

        if isinstance(obj, pd.DataFrame):
            obj.to_csv(
                file_name,
                index=False,
                encoding="utf-8-sig"
            )

            print(
                f"PASS  | {file_name} | "
                f"{len(obj):,} rows | "
                f"{os.path.getsize(file_name):,} bytes"
            )

            return True

        else:
            print(
                f"FAIL  | {file_name} | "
                f"{variable_name} exists but is not a DataFrame"
            )
            return False

    else:
        print(
            f"INFO  | {file_name} | "
            f"{variable_name} not available"
        )
        return False


# ------------------------------------------------------------
# Core artifacts
# ------------------------------------------------------------

print("\nCORE ARTIFACTS")
print("-" * 60)

save_dataframe_if_available(
    "final_test_suite",
    "final_test_suite.csv"
)

save_dataframe_if_available(
    "validation_a",
    "validation_feature_a.csv"
)

save_dataframe_if_available(
    "validation_b",
    "validation_feature_b.csv"
)

save_dataframe_if_available(
    "category_coverage",
    "category_coverage.csv"
)

save_dataframe_if_available(
    "coverage_report",
    "coverage_report.csv"
)

save_dataframe_if_available(
    "coverage_gaps",
    "coverage_gaps.csv"
)


# ------------------------------------------------------------
# Feature execution outputs
# ------------------------------------------------------------

print("\nEXECUTION ARTIFACTS")
print("-" * 60)

save_dataframe_if_available(
    "execution_feature_a",
    "execution_feature_a.csv"
)

save_dataframe_if_available(
    "execution_feature_b",
    "execution_feature_b.csv"
)


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\nFINAL SUMMARY")
print("-" * 60)

save_dataframe_if_available(
    "final_summary",
    "final_summary.csv"
)


# ------------------------------------------------------------
# Design write-up
# ------------------------------------------------------------

design_writeup = """
# Agentic AI Test Case Generator — Design Writeup

## 1. Problem Statement

The project demonstrates an Agentic AI Test Case Generator that
converts software requirements and Acceptance Criteria into
requirement-traceable QA test cases.

The project covers:

1. User Login
2. Apply Promo Code at Checkout

The generated test suites cover:

- Positive scenarios
- Negative scenarios
- Boundary scenarios
- Edge scenarios

Each test case is mapped to the relevant Acceptance Criterion.

## 2. Project Objective

The objective is to demonstrate how Agentic AI can support QA
engineers in generating comprehensive test scenarios while
maintaining requirement traceability and deterministic validation.

The overall workflow is:

Generate
→ Critique
→ Improve
→ Structure
→ Validate
→ Execute / QA Gate
→ Coverage Analysis
→ Reporting
→ Export

## 3. Architecture

The complete implementation is maintained in a single
Google Colab/Jupyter notebook.

The notebook contains:

- Common setup
- Agent implementation
- Generation logic
- Critique logic
- Improvement logic
- Test-case structuring
- Deterministic validation
- Execution / QA gate
- Coverage analysis
- Reporting
- Final artifact generation

The only external editable inputs are:

- feature_a_acceptance_criteria.json
- feature_b_acceptance_criteria.json

## 4. Feature Execution Order

Feature A — User Login is completed through its complete workflow
before Feature B begins.

Feature A includes:

Requirement
→ Acceptance Criteria
→ Generate
→ Critique
→ Improve
→ Structure
→ Validate
→ Execute / QA Gate
→ Coverage
→ Reporting
→ Final Summary

Only after Feature A is completed does Feature B start.

Feature B follows the same complete workflow.

Combined reporting occurs only after both features are completed.

## 5. Agentic Workflow

### Generator Agent

Generates the initial test suite from the requirement and Acceptance
Criteria.

The generator considers:

- Positive scenarios
- Negative scenarios
- Boundary scenarios
- Edge scenarios
- Business rules
- Thresholds
- Timing conditions
- State changes
- Priorities
- Risks
- Requirement traceability

### Critic Agent

Reviews the generated test suite and identifies:

- Missing Acceptance Criteria coverage
- Missing scenarios
- Category gaps
- Boundary gaps
- Edge cases
- Weak expected results
- Duplicate scenarios
- Traceability issues
- Unsupported assumptions

### Improver Agent

Uses the critic feedback to improve the test suite while preserving
the supplied requirements.

## 6. Deterministic Validation

Python-based deterministic validation is applied after AI generation.

Validation includes:

- Required columns
- Test Case IDs
- Duplicate IDs
- Acceptance Criteria format
- Acceptance Criteria traceability
- Scenario presence
- Test steps
- Expected results
- Categories
- Priorities
- Risks
- Category coverage
- Acceptance Criteria coverage

This provides a repeatable quality gate independent of LLM reasoning.

## 7. Test Execution

The current project does not contain a real application, API, or
browser-based System Under Test.

Therefore, the project does not fabricate functional application
PASS/FAIL results.

The execution stage performs a deterministic QA gate against the
generated test-suite artifacts.

The execution results therefore represent generated-artifact
validation rather than functional execution against a real
application.

## 8. Coverage and Traceability

Each test case is mapped to an Acceptance Criterion.

Coverage reporting identifies:

- Covered Acceptance Criteria
- Test cases mapped to each criterion
- Category coverage
- Coverage gaps

This provides requirement traceability and supports QA review.

## 9. Human Review

The Agentic AI system supports QA engineers but does not replace
human QA judgment.

Human review remains important for:

- Business intent
- Requirement ambiguity
- Risk interpretation
- Environment-specific data
- Application-specific behavior
- Automation feasibility
- Unsupported assumptions
- Final test-suite suitability

## 10. Limitations

1. AI-generated test cases require human review.
2. LLM output may vary between executions.
3. No real System Under Test is included.
4. Functional application PASS/FAIL results cannot be claimed without
   an actual application, API, or browser target.
5. Environment-specific behavior requires the target environment.
6. Deterministic validation does not replace functional testing.

## 11. Final Outputs

The notebook generates the submission artifacts from the completed
Feature A and Feature B workflows.

Expected outputs include:

- final_test_suite.csv
- validation_feature_a.csv
- validation_feature_b.csv
- category_coverage.csv
- coverage_report.csv
- coverage_gaps.csv
- execution_feature_a.csv
- execution_feature_b.csv
- final_summary.csv
- DESIGN_WRITEUP.md
- Agentic_AI_Capstone_Reflection.pdf

The implementation remains in the single notebook.

The only external editable inputs are the two Acceptance Criteria
JSON files.

## 12. Conclusion

The project demonstrates an end-to-end Agentic AI QA workflow that
transforms requirements and Acceptance Criteria into structured,
categorized, requirement-traceable test suites.

The Generate → Critique → Improve workflow provides iterative AI
assistance, while deterministic validation provides repeatable
structural quality controls.

Feature A is completed before Feature B begins, and combined
reporting occurs only after both features are complete.
"""

with open(
    "DESIGN_WRITEUP.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(design_writeup)

print(
    f"PASS  | DESIGN_WRITEUP.md | "
    f"{os.path.getsize('DESIGN_WRITEUP.md'):,} bytes"
)


# ------------------------------------------------------------
# Reflection PDF - only if already created
# ------------------------------------------------------------

if os.path.exists("Agentic_AI_Capstone_Reflection.pdf"):
    print(
        f"PASS  | Agentic_AI_Capstone_Reflection.pdf | "
        f"{os.path.getsize('Agentic_AI_Capstone_Reflection.pdf'):,} bytes"
    )
else:
    print(
        "INFO  | Agentic_AI_Capstone_Reflection.pdf | "
        "Not generated in this notebook run"
    )


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL ARTIFACT VERIFICATION")
print("=" * 60)

required_outputs = [
    "final_test_suite.csv",
    "validation_feature_a.csv",
    "validation_feature_b.csv",
    "category_coverage.csv",
    "coverage_report.csv",
    "coverage_gaps.csv",
    "DESIGN_WRITEUP.md"
]

optional_outputs = [
    "execution_feature_a.csv",
    "execution_feature_b.csv",
    "final_summary.csv",
    "Agentic_AI_Capstone_Reflection.pdf"
]

required_missing = []

for file_name in required_outputs:

    if os.path.exists(file_name):
        print(
            f"PASS  | {file_name} | "
            f"{os.path.getsize(file_name):,} bytes"
        )
    else:
        print(
            f"FAIL  | {file_name} | NOT FOUND"
        )
        required_missing.append(file_name)


print("\nOPTIONAL ARTIFACTS")
print("-" * 60)

for file_name in optional_outputs:

    if os.path.exists(file_name):
        print(
            f"PASS  | {file_name} | "
            f"{os.path.getsize(file_name):,} bytes"
        )
    else:
        print(
            f"INFO  | {file_name} | NOT GENERATED"
        )


print("\n" + "-" * 60)

if len(required_missing) == 0:
    print("✅ FINAL SUBMISSION ARTIFACT CHECK: PASS")
    print("All required artifacts are available.")
else:
    print("⚠️ FINAL SUBMISSION ARTIFACT CHECK: INCOMPLETE")
    print("Missing required files:")

    for file_name in required_missing:
        print(f"   - {file_name}")

print("=" * 60)

CREATING FINAL SUBMISSION ARTIFACTS

CORE ARTIFACTS
------------------------------------------------------------
PASS  | final_test_suite.csv | 13 rows | 4,841 bytes
FAIL  | validation_feature_a.csv | validation_a exists but is not a DataFrame
FAIL  | validation_feature_b.csv | validation_b exists but is not a DataFrame
PASS  | category_coverage.csv | 4 rows | 68 bytes
PASS  | coverage_report.csv | 21 rows | 924 bytes
PASS  | coverage_gaps.csv | 9 rows | 475 bytes

EXECUTION ARTIFACTS
------------------------------------------------------------
INFO  | execution_feature_a.csv | execution_feature_a not available
INFO  | execution_feature_b.csv | execution_feature_b not available

FINAL SUMMARY
------------------------------------------------------------
INFO  | final_summary.csv | final_summary not available
PASS  | DESIGN_WRITEUP.md | 5,512 bytes
PASS  | Agentic_AI_Capstone_Reflection.pdf | 2,792 bytes

FINAL ARTIFACT VERIFICATION
PASS  | final_test_suite.csv | 4,841 bytes
PASS  | vali

## Final Output Review

In [82]:
required_outputs = [
    "final_test_suite.csv", "validation_feature_a.csv", "validation_feature_b.csv",
    "execution_report.csv", "coverage_report.csv", "coverage_gaps.csv",
    "final_summary.csv", "category_coverage.csv",
    "Feature_A_User_Login.feature", "Feature_B_Promo_Code.feature",
    "Agentic_AI_Capstone_Reflection.pdf"
]
review = []
for filename in required_outputs:
    exists = os.path.exists(filename)
    review.append({
        "File": filename,
        "Status": "PASS" if exists and os.path.getsize(filename) > 0 else "FAIL",
        "Size": os.path.getsize(filename) if exists else 0
    })
output_review = pd.DataFrame(review)
display(output_review)
print("FINAL OUTPUT REVIEW:", "PASS" if (output_review["Status"] == "PASS").all() else "FAIL")


,File,Status,Size
0,final_test_suite.csv,PASS,4841
1,validation_feature_a.csv,PASS,881
2,validation_feature_b.csv,PASS,964
3,execution_report.csv,PASS,2062
4,coverage_report.csv,PASS,924
5,coverage_gaps.csv,PASS,475
6,final_summary.csv,PASS,288
7,category_coverage.csv,PASS,68
8,Feature_A_User_Login.feature,PASS,3628
9,Feature_B_Promo_Code.feature,PASS,1669


FINAL OUTPUT REVIEW: PASS


## Final Submission Checklist

In [83]:
submission_checks = {
    "Feature A completed before Feature B": True,
    "Feature B starts only after Feature A summary": True,
    "External Feature A AC file loaded": os.path.exists("feature_a_acceptance_criteria.json"),
    "External Feature B AC file loaded": os.path.exists("feature_b_acceptance_criteria.json"),
    "All test cases have AC IDs": bool(final_test_suite["Acceptance Criteria"].astype(str).str.match(r"^AC\d+$").all()),
    "All four categories represented": {"Positive","Negative","Boundary","Edge"}.issubset(set(final_test_suite["Category"])),
    "No duplicate Test Case IDs": final_test_suite["Test Case ID"].duplicated().sum() == 0,
    "No coverage gaps": len(coverage_gaps) == 0,
    "All generated outputs present": (output_review["Status"] == "PASS").all()
}
submission_check = pd.DataFrame([{"Check": k, "Status": "PASS" if v else "FAIL"} for k,v in submission_checks.items()])
display(submission_check)
print("PROJECT SUBMISSION CHECK:", "PASS" if (submission_check["Status"] == "PASS").all() else "FAIL")


,Check,Status
0,Feature A completed before Feature B,PASS
1,Feature B starts only after Feature A summary,PASS
2,External Feature A AC file loaded,PASS
3,External Feature B AC file loaded,PASS
4,All test cases have AC IDs,PASS
5,All four categories represented,PASS
6,No duplicate Test Case IDs,FAIL
7,No coverage gaps,FAIL
8,All generated outputs present,PASS


PROJECT SUBMISSION CHECK: FAIL
